# Tiny Aya → structured JSON from Indian bank & UPI SMS (English · Hindi · Hinglish)

**Lexsi AI Safety & Alignment Hackathon · Track 1 · Problem 5 (Structured output & extraction)**, also Problem 2 (replacing a closed-model API) and Problem 4 (low-resource / multilingual / code-mixed).

Pipeline (3 Lexsi toolkits chained):

```
smsgen (synthetic data, exact labels) ─► CuratorKIT (clean, dedup, quality gate, provenance)
      ─► AuditKIT (baselines: regex · zero-shot · few-shot) ─► AlignTune (LoRA SFT on Tiny Aya Global)
      ─► AuditKIT (after) ─► stats · error analysis · ROI ─► HF collections + CPU demo Space
```

What makes this evaluation rigorous:
1. **Held-out templates (`test_ood`)**: message formats never seen in training, so we measure generalisation, not memorisation.
2. **Hand-written challenge set**: 44 human-labelled, messy, realistic messages in all three languages.
3. **Hallucination rate**: fields the model fills in although the SMS does not contain them.
4. **Hard negatives**: OTPs, offers, bill reminders, future debits and money requests mention amounts but are *not* transactions.
5. **Ablation**: the same fine-tune *without* hard negatives or noisy data, to show what the robustness data buys.

**How to run:** Kaggle → *New Notebook* → *File → Import notebook* → this file. Settings: **GPU T4 x1**, **Internet ON**. Add a Kaggle secret `HF_TOKEN` (a Hugging Face *write* token) and accept the Tiny Aya Global licence on its model page. Then *Run all*. Set `QUICK_MODE = True` first for a ~10-minute smoke test.

In [ ]:
# ============================== CONFIG (edit me) ==============================
import os, sys, json, time, math, random, subprocess, inspect, gc, glob, shutil, traceback

HF_USERNAME = ""                      # leave empty -> taken from your HF token
PROJECT     = "tiny-aya-sms-extract"  # prefix for all HF repos
BASE_MODEL  = "CohereLabs/tiny-aya-global"
SEED        = 7

QUICK_MODE  = False   # True = tiny subsets, ~10 min end-to-end smoke test. Run this FIRST.
RUN_ABLATION = True   # 2nd fine-tune without hard negatives/noise (+~25 min)
MAKE_GGUF   = True    # export a CPU-friendly GGUF for the free demo Space
PUSH_TO_HUB = True    # create HF model/dataset repos, collections and the demo Space
SPACE_ADD_TOKEN = False  # True = store your HF token as a Space secret (only needed if GGUF export fails)

N_TRAIN, N_DEV, N_TEST_ID, N_TEST_OOD = 2400, 200, 400, 400
MAX_NEW_TOKENS = 160
GEN_BATCH      = 32
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR, EPOCHS, TRAIN_BATCH, GRAD_ACCUM, MAX_SEQ_LEN = 2e-4, 1, 4, 4, 512

# ROI assumptions (edit to your situation; sources in the article)
GPU_USD_PER_HOUR = 0.35            # typical on-demand price of a cloud T4
MONTHLY_SMS = 10_000_000           # realistic volume: a mid-size expense/lending app (~330k SMS/day)
API_PRICES = {                      # USD per 1M tokens (input, output), list prices Nov 2026
    "gpt-5-mini":   (0.25, 2.00),
    "gpt-5.4-mini": (0.75, 4.50),
}

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
for d in ["src", "data", "results", "figures", "curated", "models"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
if QUICK_MODE:
    N_TRAIN, N_DEV, N_TEST_ID, N_TEST_OOD = 160, 30, 40, 40
random.seed(SEED)
print("work dir:", WORK, "| quick mode:", QUICK_MODE)

## 1. Install the Lexsi toolkits

In [ ]:
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "FAIL ") + " ".join(args))
    if r.returncode:
        print(r.stderr[-2000:])
    return r.returncode == 0

# make the HF token visible to subprocesses (the Tiny Aya config is gated)
if not os.environ.get("HF_TOKEN"):
    for getter in ("kaggle", "colab"):
        try:
            if getter == "kaggle":
                from kaggle_secrets import UserSecretsClient
                os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
            else:
                from google.colab import userdata
                os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
            break
        except Exception:
            pass
print(sys.version, "| HF token found:", bool(os.environ.get("HF_TOKEN")))
if "transformers" in sys.modules:
    print("⚠️ transformers is already imported in this kernel. Run -> Restart kernel, then Run All again.")
OK_ALIGN   = pip("aligntune")
OK_CURATOR = pip("curatorkit")
OK_AUDIT   = pip("auditkit")
pip("peft", "trl", "accelerate", "datasets", "huggingface_hub", "matplotlib")

# Check Tiny Aya support in a SEPARATE process, so an upgrade never mixes two transformers
# versions inside this kernel (that causes "cannot import name ... from transformers.utils").
def _subprocess_check():
    code = ("import transformers, torch; from transformers import AutoConfig; "
            f"AutoConfig.from_pretrained('{BASE_MODEL}', token=__import__('os').environ.get('HF_TOKEN')); "
            "print(transformers.__version__, torch.__version__)")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return r.returncode == 0, (r.stdout or r.stderr).strip()[-600:]

ok, msg = _subprocess_check()
if not ok and "gated" not in msg.lower() and "401" not in msg:
    print("transformers can't read the Tiny Aya config -> upgrading (before importing it here):", msg[-200:])
    pip("-U", "transformers")
    ok, msg = _subprocess_check()
print("Tiny Aya config check:", "OK" if ok else "FAILED", "|", msg[-300:])

import torch, transformers   # first import in this kernel -> always the final installed version
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| CUDA", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Hugging Face login: Kaggle secret, Colab secret or env var HF_TOKEN
token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    token = token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login, whoami, HfApi
if token:
    login(token=token)
    HF_USERNAME = HF_USERNAME or whoami()["name"]
    print("logged in as", HF_USERNAME)
else:
    print("No HF token found: model download (gated) and uploads will fail.")
api = HfApi()

## 2. Project source files (generator, scoring, regex baseline, challenge set)
These are the same files as in the GitHub repo, embedded so the notebook runs on its own.

In [ ]:
# ---- src/smsgen.py ----
CONTENT = '"""Synthetic Indian bank / UPI SMS generator with exact gold labels.\n\nDesign\n------\n* Every template is a list of SEGMENTS joined by \'‖\'. The first segment is the core\n  (always kept); later segments can be dropped (optional \'?\' or by truncation). A field\n  is in the gold label ONLY if its slot survived -> the gold never contains information\n  that is not in the text (this is what lets us measure hallucination honestly).\n* \'{@inst}\' reveals the instrument without printing it (e.g. "UPI Ref" text implies UPI).\n* The LAST template of each (family, language) is held out -> `test_ood` uses message\n  formats never seen in training.\n* Bank names are FICTIONAL and messages contain no links or phone numbers, so the data\n  cannot be confused with, or reused as, real bank messages.\n"""\nfrom __future__ import annotations\n\nimport json\nimport random\nimport re\nfrom dataclasses import dataclass, field\nfrom datetime import date, timedelta\n\nFIELDS = ["is_transaction", "type", "amount", "currency", "account_last4", "instrument",\n          "counterparty", "date", "reference", "balance", "status"]\nINSTRUMENTS = ["UPI", "CARD", "NEFT", "IMPS", "RTGS", "ATM", "NACH", "AUTOPAY"]\nLANGS = ["en", "hi", "hinglish"]\n\nFICTIONAL_BANKS = ["Kestrel Bank", "Banyan Bank", "Peacock Bank", "Monsoon Bank",\n                   "Neem Bank", "Saffron Bank", "Lotus Demo Bank", "Indigo Coop Bank"]\nVPA_HANDLES = ["@kestrel", "@banyan", "@peacockpay", "@monsoon", "@neemupi", "@saffron"]\n\nMERCHANTS = ["SWIGGY", "ZOMATO", "AMAZON", "FLIPKART", "BIGBASKET", "IRCTC", "UBER", "OLA",\n             "MYNTRA", "DMART", "RELIANCE FRESH", "APOLLO PHARMACY", "BOOKMYSHOW",\n             "SHARMA GENERAL STORE", "GUPTA SWEETS", "CAFE COFFEE CORNER", "METRO CASH CARRY",\n             "HP PETROL PUMP", "DECATHLON", "NYKAA", "CROMA", "BLINKIT", "ZEPTO"]\nPERSONS = ["RAMESH KUMAR", "PRIYA SHARMA", "ANIL VERMA", "SUNITA DEVI", "ARJUN NAIR",\n           "FATIMA SHAIKH", "GURPREET SINGH", "MEERA IYER", "ROHIT DAS", "KAVYA REDDY",\n           "IMRAN KHAN", "POOJA PATEL", "VIKRAM JOSHI", "ANJALI MISHRA", "SURESH YADAV"]\nEMPLOYERS = ["ACME TECHNOLOGIES PVT LTD", "SUNRISE SOFTWARE LLP", "GANGA TEXTILES LTD",\n             "NORTHSTAR LOGISTICS", "BLUEWAVE SERVICES PVT LTD", "VEDA HOSPITALS"]\nSUBSCRIPTIONS = ["NETFLIX", "SPOTIFY", "YOUTUBE PREMIUM", "JIO FIBER", "AIRTEL POSTPAID",\n                 "TATA PLAY", "GOOGLE ONE", "HOTSTAR", "LIC PREMIUM", "SIP MUTUAL FUND"]\nBILLERS = ["CITY POWER LTD", "JAL BOARD", "METRO GAS", "AIRTEL POSTPAID", "JIO FIBER",\n           "MUNICIPAL TAX"]\nFOREIGN = [("USD", "$"), ("EUR", "€"), ("GBP", "£"), ("AED", "AED ")]\nMONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]\n\n# ---------------------------------------------------------------------------\n# Templates: family -> {"meta": {...}, "en": [...], "hi": [...], "hinglish": [...]}\n# meta: is_txn, type, status (per-template override via tuple (text, status)),\n#       inst (choices for {inst}/{@inst}), cp (counterparty pool name)\n# ---------------------------------------------------------------------------\nT = {\n    "upi_debit": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["UPI"], cp="upi_party"),\n        "en": [\n            "Your A/c {acct} is debited for {amt}‖ on {date}‖ towards {cp}‖ via {inst}‖. UPI Ref No {ref}‖. Avl Bal {bal}",\n            "{bank}: {amt} debited from a/c {acct}‖ on {date}‖ to VPA {cp}{@inst}‖ (UPI Ref {ref})‖. Avl bal {bal}",\n            "Sent {amt} from {bank} A/c {acct} to {cp} on {date}{@inst}‖. Ref: {ref}‖. Bal: {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} से {amt} डेबिट किए गए‖ दिनांक {date}‖, {cp} को‖ {inst} द्वारा‖। UPI संदर्भ {ref}‖। उपलब्ध शेष {bal}",\n            "{bank}: खाता {acct} से {amt} {cp} को भेजे गए{@inst}‖ ({date})‖। संदर्भ संख्या {ref}‖। शेष राशि {bal}",\n            "{amt} का UPI भुगतान{@inst} {cp} को सफल रहा, खाता {acct}‖, दिनांक {date}‖। रेफ़ {ref}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} se {amt} debit hue‖ {date} ko‖ {cp} ko‖ {inst} se‖. UPI Ref {ref}‖. Avl bal {bal}",\n            "{bank}: A/c {acct} se {amt} {cp} ko bheje gaye{@inst}‖ ({date})‖. Ref no {ref}‖. Bal {bal}",\n            "{amt} ka UPI payment{@inst} {cp} ko successful, A/c {acct}‖, date {date}‖. Ref {ref}",\n        ],\n    },\n    "upi_credit": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=["UPI"], cp="upi_party"),\n        "en": [\n            "Dear Customer, your A/c {acct} is credited with {amt}‖ on {date}‖ by {cp}‖ via {inst}‖. Ref {ref}‖. Avl Bal {bal}",\n            "{amt} received in {bank} a/c {acct} from {cp}{@inst}‖ on {date}‖. UPI Ref:{ref}",\n            "Money received! {cp} sent you {amt} via {inst}‖ on {date}‖. Credited to A/c {acct}‖. Bal {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} में {amt} जमा किए गए‖ दिनांक {date}‖, {cp} द्वारा‖ {inst} से‖। संदर्भ {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} ने आपको {amt} भेजे{@inst}‖, खाता {acct} में जमा‖ ({date})‖। UPI रेफ़ {ref}",\n            "खाता {acct} में {cp} से {amt} प्राप्त हुए‖ {date} को‖ {inst} द्वारा‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} mein {amt} credit hue‖ {date} ko‖ {cp} se‖ {inst} dwara‖. Ref {ref}‖. Avl bal {bal}",\n            "{cp} ne aapko {amt} bheje{@inst}‖, A/c {acct} mein jama‖ ({date})‖. UPI Ref {ref}",\n            "A/c {acct} mein {cp} se {amt} mile‖ {date} ko‖ {inst} se‖. Bal {bal}",\n        ],\n    },\n    "card_spend": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["CARD"], cp="merchant", foreign=0.25),\n        "en": [\n            "You have spent {amt} on your {bank} Credit Card {card}‖ at {cp}‖ on {date}‖. Avl limit: {limit}",\n            "Txn of {amt} on Card {card} at {cp}‖ on {date}‖. Ref {ref}‖. Available credit limit {limit}",\n            "{bank} Debit Card {card} used for {amt} at {cp}‖ on {date} {time}‖. Avl bal {bal}",\n        ],\n        "hi": [\n            "आपके {bank} क्रेडिट कार्ड {card} से {cp} पर {amt} खर्च किए गए‖ दिनांक {date}‖। उपलब्ध लिमिट {limit}",\n            "कार्ड {card} से {amt} का लेनदेन {cp} पर‖, {date}‖। संदर्भ {ref}‖। उपलब्ध क्रेडिट लिमिट {limit}",\n            "{bank} डेबिट कार्ड {card} का उपयोग {cp} पर {amt} के लिए किया गया‖, {date} {time}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke {bank} Credit Card {card} se {cp} par {amt} kharch hue‖ {date} ko‖. Avl limit {limit}",\n            "Card {card} se {amt} ka txn {cp} par‖, {date}‖. Ref {ref}‖. Avl credit limit {limit}",\n            "{bank} Debit Card {card} use hua {cp} par {amt} ke liye‖, {date} {time}‖. Bal {bal}",\n        ],\n    },\n    "atm_withdrawal": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["ATM"], cp=None),\n        "en": [\n            "{amt} withdrawn from A/c {acct}‖ at {inst}‖ on {date}‖. Avl Bal {bal}",\n            "Cash withdrawal{@inst} of {amt} from your {bank} A/c {acct}‖ on {date} {time}‖. Bal: {bal}",\n            "ATM txn{@inst}: {amt} debited from a/c {acct}‖ on {date}‖. Avl bal {bal}",\n        ],\n        "hi": [\n            "खाता {acct} से {amt} की नकद निकासी{@inst}‖ {date} को‖। उपलब्ध शेष {bal}",\n            "{inst} से {amt} निकाले गए, खाता {acct}‖, दिनांक {date}‖। शेष राशि {bal}",\n            "{bank} ATM लेनदेन{@inst}: खाता {acct} से {amt} डेबिट‖, {date} {time}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "A/c {acct} se {amt} cash nikale gaye{@inst}‖ {date} ko‖. Avl bal {bal}",\n            "{inst} se {amt} withdraw hue, A/c {acct}‖, date {date}‖. Bal {bal}",\n            "{bank} ATM txn{@inst}: A/c {acct} se {amt} debit‖, {date} {time}‖. Bal {bal}",\n        ],\n    },\n    "transfer_credit": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=["NEFT", "IMPS", "RTGS"], cp="transfer_party"),\n        "en": [\n            "Your A/c {acct} has been credited with {amt}‖ on {date}‖ by {inst}‖ from {cp}‖. UTR {ref}‖. Avl Bal {bal}",\n            "{inst} credit of {amt} received in A/c {acct} from {cp}‖ on {date}‖. Ref {ref}",\n            "Salary credited: {amt} to A/c {acct}‖ from {cp}‖ via {inst}‖ on {date}‖. Bal {bal}",\n        ],\n        "hi": [\n            "आपके खाते {acct} में {inst} द्वारा {amt} जमा हुए‖, {cp} से‖, दिनांक {date}‖। UTR {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} से {amt} का {inst} क्रेडिट खाता {acct} में प्राप्त‖ ({date})‖। संदर्भ {ref}",\n            "वेतन जमा: खाता {acct} में {amt}‖, {cp} द्वारा‖ {inst} से‖, {date}‖। शेष {bal}",\n        ],\n        "hinglish": [\n            "Aapke A/c {acct} mein {inst} se {amt} credit hue‖ {cp} se‖, {date} ko‖. UTR {ref}‖. Avl bal {bal}",\n            "{cp} se {amt} ka {inst} credit A/c {acct} mein aaya‖ ({date})‖. Ref {ref}",\n            "Salary aa gayi: A/c {acct} mein {amt}‖ {cp} se‖ {inst} dwara‖, {date}‖. Bal {bal}",\n        ],\n    },\n    "transfer_debit": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["NEFT", "IMPS", "RTGS"], cp="person"),\n        "en": [\n            "{amt} debited from A/c {acct}‖ on {date}‖ for {inst} transfer‖ to {cp}‖. Ref {ref}‖. Avl Bal {bal}",\n            "Your {inst} of {amt} to {cp} from A/c {acct} is successful‖. Ref No {ref}‖. Date {date}",\n            "A/c {acct} debited by {amt}‖ on {date}‖; {inst} to {cp}‖, UTR {ref}",\n        ],\n        "hi": [\n            "खाता {acct} से {amt} डेबिट‖, {date}‖, {inst} ट्रांसफ़र‖ {cp} को‖। संदर्भ {ref}‖। उपलब्ध शेष {bal}",\n            "{cp} को {amt} का आपका {inst} ट्रांसफ़र सफल रहा, खाता {acct}‖। संदर्भ संख्या {ref}‖। दिनांक {date}",\n            "खाता {acct} से {amt} कटे‖ {date} को‖; {cp} को {inst}‖, UTR {ref}",\n        ],\n        "hinglish": [\n            "A/c {acct} se {amt} debit‖, {date}‖, {inst} transfer‖ {cp} ko‖. Ref {ref}‖. Avl bal {bal}",\n            "{cp} ko {amt} ka aapka {inst} transfer successful, A/c {acct}‖. Ref no {ref}‖. Date {date}",\n            "A/c {acct} se {amt} kat gaye‖ {date} ko‖; {cp} ko {inst}‖, UTR {ref}",\n        ],\n    },\n    "refund_reversal": {\n        "meta": dict(is_txn=True, type="credit", status="success", inst=[], cp="merchant"),\n        "en": [\n            ("Refund of {amt} from {cp} credited to your A/c {acct}‖ on {date}‖. Avl Bal {bal}", "success"),\n            ("{amt} debited on {date} for txn to {cp} has been reversed to your A/c {acct}‖. Ref {ref}", "reversed"),\n            ("Your refund of {amt} for order at {cp} is processed to Card {card}‖ on {date}", "success"),\n        ],\n        "hi": [\n            ("{cp} से {amt} का रिफ़ंड आपके खाते {acct} में जमा किया गया‖, दिनांक {date}‖। उपलब्ध शेष {bal}", "success"),\n            ("{date} को {cp} के लेनदेन में कटे {amt} आपके खाते {acct} में वापस कर दिए गए हैं‖। संदर्भ {ref}", "reversed"),\n            ("{cp} पर ऑर्डर का {amt} रिफ़ंड कार्ड {card} में भेज दिया गया है‖, {date}", "success"),\n        ],\n        "hinglish": [\n            ("{cp} se {amt} ka refund aapke A/c {acct} mein credit hua‖ {date} ko‖. Avl bal {bal}", "success"),\n            ("{date} ko {cp} txn mein kate {amt} aapke A/c {acct} mein wapas aa gaye‖. Ref {ref}", "reversed"),\n            ("{cp} order ka {amt} refund Card {card} mein bhej diya gaya‖, {date}", "success"),\n        ],\n    },\n    "failed_txn": {\n        "meta": dict(is_txn=True, type="debit", status="failed", inst=["UPI"], cp="merchant"),\n        "en": [\n            "Txn of {amt} to {cp} via {inst} FAILED‖ on {date}‖. Amount, if debited, will be refunded‖. Ref {ref}",\n            "Your payment of {amt} to {cp} could not be processed{@inst}‖ (UPI Ref {ref})‖. A/c {acct} not debited",\n            "Transaction declined: {amt} at {cp} on Card {card}‖ on {date}‖. Reason: insufficient balance",\n        ],\n        "hi": [\n            "{inst} द्वारा {cp} को {amt} का लेनदेन विफल रहा‖, दिनांक {date}‖। अगर राशि कटी है तो वापस आ जाएगी‖। संदर्भ {ref}",\n            "{cp} को {amt} का आपका भुगतान नहीं हो सका{@inst}‖ (UPI रेफ़ {ref})‖। खाता {acct} से राशि नहीं कटी",\n            "लेनदेन अस्वीकृत: कार्ड {card} से {cp} पर {amt}‖, {date}‖। कारण: अपर्याप्त शेष",\n        ],\n        "hinglish": [\n            "{inst} se {cp} ko {amt} ka txn fail ho gaya‖ {date} ko‖. Paisa kata hai to wapas aa jayega‖. Ref {ref}",\n            "{cp} ko {amt} ka aapka payment nahi ho paya{@inst}‖ (UPI Ref {ref})‖. A/c {acct} se paisa nahi kata",\n            "Txn decline hua: Card {card} se {cp} par {amt}‖, {date}‖. Reason: balance kam hai",\n        ],\n    },\n    "autopay_emi": {\n        "meta": dict(is_txn=True, type="debit", status="success", inst=["NACH", "AUTOPAY"], cp="subscription"),\n        "en": [\n            "{amt} debited from A/c {acct} towards EMI for loan {ref}‖ via {inst}‖ on {date}‖. Avl Bal {bal}",\n            "AutoPay{@inst=AUTOPAY} of {amt} for {cp} executed from A/c {acct}‖ on {date}‖. Ref {ref}",\n            "Your {cp} subscription of {amt} was paid via {inst}‖ from A/c {acct}‖ on {date}",\n        ],\n        "hi": [\n            "लोन {ref} की EMI के लिए खाता {acct} से {amt} डेबिट‖ {inst} द्वारा‖, दिनांक {date}‖। उपलब्ध शेष {bal}",\n            "{cp} के लिए {amt} का ऑटोपे{@inst=AUTOPAY} खाता {acct} से पूरा हुआ‖, {date}‖। संदर्भ {ref}",\n            "आपकी {cp} सदस्यता के {amt} {inst} से चुकाए गए‖, खाता {acct}‖, {date}",\n        ],\n        "hinglish": [\n            "Loan {ref} ki EMI ke liye A/c {acct} se {amt} debit‖ {inst} se‖, {date} ko‖. Avl bal {bal}",\n            "{cp} ke liye {amt} ka AutoPay{@inst=AUTOPAY} A/c {acct} se ho gaya‖, {date}‖. Ref {ref}",\n            "Aapki {cp} subscription ke {amt} {inst} se pay hue‖, A/c {acct}‖, {date}",\n        ],\n    },\n    # ---------------- hard negatives: money mentioned, but nothing moved ----------------\n    "otp": {\n        "meta": dict(is_txn=False, cp="merchant"),\n        "en": [\n            "{otp} is the OTP for your txn of {amt} at {cp} on {bank} Card {card}. Valid for 5 mins. Do not share it with anyone.",\n            "OTP {otp} to authorise payment of {amt} to {cp}. Never share your OTP. -{bank}",\n        ],\n        "hi": [\n            "{otp} आपके {bank} कार्ड {card} से {cp} पर {amt} के लेनदेन का OTP है। 5 मिनट तक मान्य। इसे किसी से साझा न करें।",\n            "{cp} को {amt} के भुगतान के लिए OTP {otp} है। अपना OTP कभी किसी को न बताएँ। -{bank}",\n        ],\n        "hinglish": [\n            "{otp} aapke {bank} Card {card} se {cp} par {amt} ke txn ka OTP hai. 5 min tak valid. Kisi ke saath share na karein.",\n            "{cp} ko {amt} payment ke liye OTP {otp} hai. Apna OTP kabhi kisi ko na batayein. -{bank}",\n        ],\n    },\n    "promo": {\n        "meta": dict(is_txn=False, cp=None),\n        "en": [\n            "Get cashback up to {amt} on your next UPI payment with {bank}! Offer valid till {date}. T&C apply.",\n            "Pre-approved loan of up to {amt} is waiting for you. Apply in the {bank} app today.",\n        ],\n        "hi": [\n            "{bank} के साथ अगले UPI भुगतान पर {amt} तक का कैशबैक पाएँ! ऑफ़र {date} तक मान्य। नियम व शर्तें लागू।",\n            "आपके लिए {amt} तक का प्री-अप्रूव्ड लोन तैयार है। आज ही {bank} ऐप में आवेदन करें।",\n        ],\n        "hinglish": [\n            "{bank} ke saath agle UPI payment par {amt} tak cashback paayein! Offer {date} tak valid. T&C apply.",\n            "Aapke liye {amt} tak ka pre-approved loan ready hai. Aaj hi {bank} app mein apply karein.",\n        ],\n    },\n    "bill_due": {\n        "meta": dict(is_txn=False, cp="biller"),\n        "en": [\n            "Your {bank} Credit Card {card} bill of {amt} is due on {date}. Minimum due {amt2}. Please pay on time to avoid charges.",\n            "Reminder: {cp} bill of {amt} is due on {date}. Ignore if already paid.",\n        ],\n        "hi": [\n            "आपके {bank} क्रेडिट कार्ड {card} का {amt} का बिल {date} को देय है। न्यूनतम देय राशि {amt2}। शुल्क से बचने के लिए समय पर भुगतान करें।",\n            "रिमाइंडर: {cp} का {amt} का बिल {date} को देय है। यदि भुगतान कर दिया है तो इसे अनदेखा करें।",\n        ],\n        "hinglish": [\n            "Aapke {bank} Credit Card {card} ka {amt} ka bill {date} ko due hai. Minimum due {amt2}. Charges se bachne ke liye time par pay karein.",\n            "Reminder: {cp} ka {amt} ka bill {date} ko due hai. Pay kar diya hai to ignore karein.",\n        ],\n    },\n    "scheduled_debit": {\n        "meta": dict(is_txn=False, cp="subscription"),\n        "en": [\n            "{amt} will be debited from A/c {acct} on {date} towards {cp} AutoPay. Please maintain sufficient balance.",\n            "Upcoming EMI of {amt} for loan {ref} will be deducted on {date}. -{bank}",\n        ],\n        "hi": [\n            "{date} को {cp} ऑटोपे के लिए खाता {acct} से {amt} डेबिट किए जाएँगे। कृपया पर्याप्त शेष रखें।",\n            "लोन {ref} की {amt} की आगामी EMI {date} को कटेगी। -{bank}",\n        ],\n        "hinglish": [\n            "{date} ko {cp} AutoPay ke liye A/c {acct} se {amt} debit honge. Kripya sufficient balance rakhein.",\n            "Loan {ref} ki {amt} ki agli EMI {date} ko kategi. -{bank}",\n        ],\n    },\n    "balance_info": {\n        "meta": dict(is_txn=False, cp=None),\n        "en": [\n            "Avl bal in your {bank} A/c {acct} is {bal} as on {date}.",\n            "Dear Customer, the balance in A/c {acct} on {date} is {bal}. -{bank}",\n        ],\n        "hi": [\n            "{date} को आपके {bank} खाते {acct} में उपलब्ध शेष {bal} है।",\n            "प्रिय ग्राहक, खाता {acct} में {date} को शेष राशि {bal} है। -{bank}",\n        ],\n        "hinglish": [\n            "{date} ko aapke {bank} A/c {acct} mein avl bal {bal} hai.",\n            "Dear Customer, A/c {acct} mein {date} ko balance {bal} hai. -{bank}",\n        ],\n    },\n}\nTXN_FAMILIES = [f for f, v in T.items() if v["meta"]["is_txn"]]\nNEG_FAMILIES = [f for f, v in T.items() if not v["meta"]["is_txn"]]\n\n\n# ---------------------------------------------------------------------------\n# Value renderers\n# ---------------------------------------------------------------------------\ndef indian_group(n: int) -> str:\n    s = str(n)\n    if len(s) <= 3:\n        return s\n    head, tail = s[:-3], s[-3:]\n    parts = []\n    while len(head) > 2:\n        parts.insert(0, head[-2:])\n        head = head[:-2]\n    if head:\n        parts.insert(0, head)\n    return ",".join(parts) + "," + tail\n\n\ndef fmt_number(v: float, rng: random.Random) -> str:\n    whole = int(round(v * 100)) // 100\n    paise = int(round(v * 100)) % 100\n    style = rng.choice(["plain", "indian", "indian", "western"])\n    w = str(whole) if style == "plain" else indian_group(whole) if style == "indian" else f"{whole:,}"\n    if paise == 0 and rng.random() < 0.5:\n        return w\n    return f"{w}.{paise:02d}"\n\n\nINR_FORMATS = {\n    "en": ["Rs.{n}", "Rs {n}", "INR {n}", "₹{n}", "Rs.{n}/-", "INR{n}"],\n    "hi": ["₹{n}", "रु. {n}", "रुपये {n}", "Rs.{n}", "₹ {n}"],\n    "hinglish": ["Rs {n}", "Rs.{n}", "₹{n}", "{n} rupaye", "INR {n}"],\n}\n\n\ndef rand_amount(rng: random.Random, lo=1.0, hi=500000.0) -> float:\n    import math\n    v = math.exp(rng.uniform(math.log(lo), math.log(hi)))\n    return round(v, 2) if rng.random() < 0.6 else float(int(v))\n\n\ndef fmt_inr(v: float, lang: str, rng: random.Random) -> str:\n    return rng.choice(INR_FORMATS[lang]).format(n=fmt_number(v, rng))\n\n\ndef fmt_foreign(v: float, code: str, sym: str, rng: random.Random) -> str:\n    n = f"{v:.2f}"\n    return rng.choice([f"{code} {n}", f"{sym}{n}" if sym.strip() != code else f"{code} {n}", f"{code}{n}"])\n\n\ndef fmt_date(d: date, rng: random.Random) -> str:\n    style = rng.randrange(8)\n    m = MONTHS[d.month - 1]\n    return [\n        f"{d.day:02d}-{d.month:02d}-{d.year % 100:02d}",\n        f"{d.day:02d}/{d.month:02d}/{d.year}",\n        f"{d.day:02d}{m}{d.year % 100:02d}",\n        f"{d.day:02d}-{m}-{d.year}",\n        f"{d.year}-{d.month:02d}-{d.day:02d}",\n        f"{d.day:02d}.{d.month:02d}.{d.year}",\n        f"{d.day} {m} {d.year}",\n        f"{d.day:02d}-{m.upper()}-{d.year % 100:02d}",\n    ][style]\n\n\ndef fmt_acct(l4: str, lang: str, rng: random.Random) -> str:\n    opts = [f"XX{l4}", f"XXXXXX{l4}", f"*{l4}", f"x{l4}", f"XXXX{l4}", f"**{l4}"]\n    if lang == "en":\n        opts += [f"no. XX{l4}", f"ending {l4}"]\n    return rng.choice(opts)\n\n\ndef fmt_card(l4: str, rng: random.Random) -> str:\n    return rng.choice([f"XX{l4}", f"ending {l4}", f"**** {l4}", f"xx{l4}", f"XXXX-{l4}"])\n\n\ndef rand_ref(inst: str | None, rng: random.Random) -> str:\n    if inst in ("NEFT", "RTGS") and rng.random() < 0.6:\n        return rng.choice("NHSKBP") + "".join(rng.choice("0123456789") for _ in range(15))\n    if rng.random() < 0.15:\n        return "".join(rng.choice("ABCDEFGHJKLMNPQRSTUVWXYZ0123456789") for _ in range(10))\n    return "".join(rng.choice("0123456789") for _ in range(12))\n\n\ndef rand_counterparty(pool: str | None, rng: random.Random) -> str | None:\n    if pool is None:\n        return None\n    if pool == "upi_party":\n        r = rng.random()\n        if r < 0.4:\n            return rng.choice(MERCHANTS)\n        if r < 0.7:\n            return rng.choice(PERSONS)\n        name = rng.choice(PERSONS).lower().split()\n        return f"{name[0]}.{name[1][:1]}{rng.randint(1, 99)}{rng.choice(VPA_HANDLES)}"\n    if pool == "transfer_party":\n        return rng.choice(EMPLOYERS) if rng.random() < 0.5 else rng.choice(PERSONS)\n    return rng.choice({"merchant": MERCHANTS, "person": PERSONS, "subscription": SUBSCRIPTIONS,\n                       "biller": BILLERS}[pool])\n\n\n# ---------------------------------------------------------------------------\n# Perturbations (edge cases). Each returns (text, tag) and never changes gold values.\n# ---------------------------------------------------------------------------\nDEV_DIGITS = str.maketrans("0123456789", "०१२३४५६७८९")\n\n\ndef perturb(text: str, lang: str, rng: random.Random, p: float = 0.35) -> tuple[str, list[str]]:\n    tags = []\n    if rng.random() > p:\n        return text, tags\n    ops = ["spacing", "header", "promo_tail", "footer"]\n    if lang in ("en", "hinglish"):\n        ops += ["upper", "lower"]\n    if lang == "hi":\n        ops += ["dev_digits"]\n    for op in rng.sample(ops, k=rng.choice([1, 1, 2])):\n        if op == "upper":\n            text = text.upper()\n        elif op == "lower":\n            text = text.lower()\n        elif op == "spacing":\n            # remove a space between a number and a letter, or double a space\n            if rng.random() < 0.5:\n                text = re.sub(r"(?<=\\d) (?=[^\\W\\d])", "", text, count=1)\n            else:\n                text = re.sub(r" ", "  ", text, count=1)\n        elif op == "header":\n            text = rng.choice(["[Kestrel Bk] ", "BNYN: ", "Alert: ", "Txn Alert- "]) + text\n        elif op == "footer":\n            text = text + rng.choice([" -Kestrel Bank", " Not you? Report via your bank app.",\n                                      " Download our app for details.", " - Neem Bank"])\n        elif op == "promo_tail":\n            text = text + rng.choice([" Get 5% cashback up to Rs.100 on your next txn!",\n                                      " Earn 2X reward points on weekend spends.",\n                                      " Win vouchers worth Rs 500. T&C apply."])\n        elif op == "dev_digits":\n            text = text.translate(DEV_DIGITS)\n        tags.append(op)\n    return text, tags\n\n\n# ---------------------------------------------------------------------------\n# Rendering one example\n# ---------------------------------------------------------------------------\n@dataclass\nclass Example:\n    id: str\n    family: str\n    lang: str\n    template_id: str\n    sms: str\n    gold: dict\n    tags: list = field(default_factory=list)\n    split: str = ""\n\n    def to_json(self) -> dict:\n        return {"id": self.id, "family": self.family, "lang": self.lang,\n                "template_id": self.template_id, "sms": self.sms,\n                "gold": self.gold, "tags": self.tags, "split": self.split}\n\n\n_SLOT = re.compile(r"\\{(@?)(\\w+)(?:=(\\w+))?\\}")\n\n\ndef render(family: str, lang: str, t_idx: int, rng: random.Random, ex_id: str,\n           noise_p: float = 0.35, allow_drop: bool = True) -> Example:\n    spec = T[family]\n    meta = spec["meta"]\n    tmpl = spec[lang][t_idx]\n    status = meta.get("status")\n    if isinstance(tmpl, tuple):\n        tmpl, status = tmpl\n    segments = tmpl.split("‖")\n    keep = [True] * len(segments)\n    tags = []\n    if allow_drop and meta["is_txn"] and len(segments) > 1:\n        # optional-segment dropping (30%) and truncation (10%)\n        for k in range(1, len(segments)):\n            if rng.random() < 0.15:\n                keep[k] = False\n        if any(not x for x in keep[1:]):\n            tags.append("missing_fields")\n        if rng.random() < 0.10:\n            cut = rng.randint(1, len(segments) - 1)\n            for k in range(cut, len(segments)):\n                keep[k] = False\n            tags.append("truncated")\n\n    inst = rng.choice(meta["inst"]) if meta.get("inst") else None\n    foreign = meta.get("foreign", 0) and rng.random() < meta["foreign"]\n    amount = rand_amount(rng, 1, 3000) if foreign else rand_amount(rng)\n    code, sym = rng.choice(FOREIGN) if foreign else ("INR", "₹")\n    d = date(2025, 1, 1) + timedelta(days=rng.randrange(0, 730))\n    acct = f"{rng.randint(0, 9999):04d}"\n    card = f"{rng.randint(0, 9999):04d}"\n    values = {\n        "amt": fmt_foreign(amount, code, sym, rng) if foreign else fmt_inr(amount, lang, rng),\n        "amt2": fmt_inr(round(amount * 0.05, 2) if amount > 100 else amount, lang, rng),\n        "acct": fmt_acct(acct, lang, rng),\n        "card": fmt_card(card, rng),\n        "date": fmt_date(d, rng),\n        "time": f"{rng.randint(0, 23):02d}:{rng.randint(0, 59):02d}",\n        "cp": rand_counterparty(meta.get("cp"), rng) or "",\n        "ref": rand_ref(inst, rng),\n        "bal": fmt_inr(rand_amount(rng, 10, 900000), lang, rng),\n        "limit": fmt_inr(rand_amount(rng, 1000, 500000), lang, rng),\n        "inst": {"AUTOPAY": "AutoPay"}.get(inst, inst) if inst else "",\n        "bank": rng.choice(FICTIONAL_BANKS),\n        "otp": f"{rng.randint(0, 999999):06d}",\n    }\n    gold_values = {\n        "amt": ("amount", amount), "acct": ("account_last4", acct), "card": ("account_last4", card),\n        "date": ("date", d.isoformat()), "cp": ("counterparty", values["cp"] or None),\n        "ref": ("reference", values["ref"]), "bal": ("balance", None),\n        "inst": ("instrument", inst),\n    }\n    # balance gold value is the numeric value we rendered; re-derive from text below\n    bal_value = None\n\n    revealed = {}\n    parts = []\n    for seg, k in zip(segments, keep):\n        if not k:\n            continue\n\n        def sub(m):\n            hidden, name, fixed = m.group(1), m.group(2), m.group(3)\n            if name in gold_values:\n                fname, val = gold_values[name]\n                if fixed:\n                    val = fixed\n                revealed[fname] = val\n                if name == "card":\n                    revealed["instrument"] = "CARD"\n                if name == "amt":\n                    revealed["currency"] = code\n            if name == "bal":\n                revealed["balance"] = "__BAL__"\n            return "" if hidden else values[name]\n        parts.append(_SLOT.sub(sub, seg))\n    text = "".join(parts).strip()\n    # numeric balance value from rendered string\n    if "balance" in revealed:\n        bal_value = parse_rendered_number(values["bal"])\n        revealed["balance"] = bal_value\n    if text and text[-1] not in ".।!":\n        text += "." if lang != "hi" else "।"\n\n    gold = {f: None for f in FIELDS}\n    gold["is_transaction"] = bool(meta["is_txn"])\n    if meta["is_txn"]:\n        gold["type"] = meta["type"]\n        gold["status"] = status\n        for k, v in revealed.items():\n            gold[k] = v\n        if foreign:\n            tags.append("foreign_currency")\n        if "{limit}" in "".join(s for s, k in zip(segments, keep) if k):\n            tags.append("limit_distractor")\n    else:\n        tags.append("hard_negative")\n    text, ptags = perturb(text, lang, rng, p=noise_p)\n    tags += ptags\n    return Example(id=ex_id, family=family, lang=lang, template_id=f"{family}/{lang}/{t_idx}",\n                   sms=text, gold=gold, tags=tags)\n\n\n_NUM = re.compile(r"[\\d,]+(?:\\.\\d+)?")\n\n\ndef parse_rendered_number(s: str) -> float:\n    m = _NUM.search(s)\n    return float(m.group(0).replace(",", "")) if m else None\n\n\n# ---------------------------------------------------------------------------\n# Dataset builder with template-level held-out split\n# ---------------------------------------------------------------------------\ndef build_dataset(n_train=2400, n_dev=200, n_test_id=400, n_test_ood=400, seed=7,\n                  neg_share=0.3, noise_p=0.35, allow_drop=True,\n                  exclude_sms: set | None = None) -> dict[str, list[Example]]:\n    """noise_p / allow_drop / neg_share = 0 give the \'clean data\' ablation."""\n    rng = random.Random(seed)\n    seen_sms = set(exclude_sms or ())\n    out = {"train": [], "dev": [], "test_id": [], "test_ood": []}\n\n    def pick(held_out: bool):\n        fam = rng.choice(NEG_FAMILIES) if rng.random() < neg_share else rng.choice(TXN_FAMILIES)\n        lang = rng.choice(LANGS)\n        n_t = len(T[fam][lang])\n        t_idx = n_t - 1 if held_out else rng.randrange(0, n_t - 1)\n        return fam, lang, t_idx\n\n    for split, n in [("test_ood", n_test_ood), ("test_id", n_test_id), ("dev", n_dev), ("train", n_train)]:\n        i = 0\n        tries = 0\n        while len(out[split]) < n:\n            tries += 1\n            fam, lang, t_idx = pick(held_out=(split == "test_ood"))\n            ex = render(fam, lang, t_idx, rng, f"{split}-{i:05d}", noise_p=noise_p,\n                        allow_drop=allow_drop)\n            if ex.sms in seen_sms:      # exact-duplicate guard across ALL splits (no leakage)\n                continue\n            seen_sms.add(ex.sms)\n            ex.split = split\n            out[split].append(ex)\n            i += 1\n    return out\n\n\ndef gold_to_str(gold: dict) -> str:\n    return json.dumps({k: gold[k] for k in FIELDS}, ensure_ascii=False)\n\n\nSYSTEM_PROMPT = (\n    "You extract structured data from Indian bank/UPI SMS (English, Hindi or Hinglish). "\n    "Reply with ONLY one JSON object with exactly these keys: "\n    "is_transaction (true only if money actually moved or a transaction was attempted/failed/reversed; "\n    "false for OTPs, offers, reminders, future debits, balance info), "\n    "type (\\"debit\\"|\\"credit\\"|null), amount (number), currency (ISO code, e.g. \\"INR\\"), "\n    "account_last4 (last 4 digits as string), "\n    "instrument (\\"UPI\\"|\\"CARD\\"|\\"NEFT\\"|\\"IMPS\\"|\\"RTGS\\"|\\"ATM\\"|\\"NACH\\"|\\"AUTOPAY\\"|null), "\n    "counterparty (merchant/person/VPA as written), date (YYYY-MM-DD; SMS dates are day-first), "\n    "reference (string), balance (number; available balance only, never credit limit), "\n    "status (\\"success\\"|\\"failed\\"|\\"reversed\\"|null). "\n    "Use null for anything not stated in the SMS. Never guess. Use ASCII digits. "\n    "If is_transaction is false, set every other key to null."\n)\n\n\ndef build_messages(sms: str, few_shot: list[tuple[str, str]] | None = None) -> list[dict]:\n    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]\n    for s, g in (few_shot or []):\n        msgs.append({"role": "user", "content": f"SMS: {s}"})\n        msgs.append({"role": "assistant", "content": g})\n    msgs.append({"role": "user", "content": f"SMS: {sms}"})\n    return msgs\n\n\nif __name__ == "__main__":\n    ds = build_dataset()\n    for k, v in ds.items():\n        print(k, len(v))\n    for ex in ds["train"][:6]:\n        print(ex.lang, ex.family, ex.tags, "\\n ", ex.sms, "\\n ", gold_to_str(ex.gold))\n'
open(f"{WORK}/src/smsgen.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/smsgen.py', len(CONTENT), 'chars')

In [ ]:
# ---- src/scoring.py ----
CONTENT = '"""Parsing, schema validation and metrics for SMS -> JSON extraction."""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport random\nimport re\nimport unicodedata\nfrom collections import defaultdict\n\nFIELDS = ["is_transaction", "type", "amount", "currency", "account_last4", "instrument",\n          "counterparty", "date", "reference", "balance", "status"]\nVALUE_FIELDS = FIELDS[1:]\nENUMS = {\n    "type": {"debit", "credit"},\n    "instrument": {"UPI", "CARD", "NEFT", "IMPS", "RTGS", "ATM", "NACH", "AUTOPAY"},\n    "status": {"success", "failed", "reversed"},\n}\n_DEV = str.maketrans("०१२३४५६७८९", "0123456789")\n\n\n# ---------------------------------------------------------------------------\n# Parsing\n# ---------------------------------------------------------------------------\ndef extract_json(text: str) -> dict | None:\n    """Return the first JSON object in `text` (tolerates code fences / chatter)."""\n    if not text:\n        return None\n    t = text.strip()\n    t = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", t)\n    start = t.find("{")\n    while start != -1:\n        depth, in_str, esc = 0, False, False\n        for i in range(start, len(t)):\n            c = t[i]\n            if in_str:\n                if esc:\n                    esc = False\n                elif c == "\\\\":\n                    esc = True\n                elif c == \'"\':\n                    in_str = False\n            elif c == \'"\':\n                in_str = True\n            elif c == "{":\n                depth += 1\n            elif c == "}":\n                depth -= 1\n                if depth == 0:\n                    try:\n                        obj = json.loads(t[start:i + 1])\n                        return obj if isinstance(obj, dict) else None\n                    except json.JSONDecodeError:\n                        break\n        start = t.find("{", start + 1)\n    return None\n\n\ndef schema_errors(obj: dict | None) -> list[str]:\n    if obj is None:\n        return ["not_json"]\n    errs = []\n    keys = set(obj)\n    if keys != set(FIELDS):\n        if set(FIELDS) - keys:\n            errs.append("missing_keys")\n        if keys - set(FIELDS):\n            errs.append("extra_keys")\n    if not isinstance(obj.get("is_transaction"), bool):\n        errs.append("bad_is_transaction")\n    for k, allowed in ENUMS.items():\n        v = obj.get(k)\n        if v is not None and (not isinstance(v, str) or v not in allowed):\n            errs.append(f"bad_{k}")\n    for k in ("amount", "balance"):\n        v = obj.get(k)\n        if v is not None and not (isinstance(v, (int, float)) and not isinstance(v, bool)):\n            errs.append(f"bad_{k}")\n    d = obj.get("date")\n    if d is not None and not (isinstance(d, str) and re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", d)):\n        errs.append("bad_date")\n    a = obj.get("account_last4")\n    if a is not None and not (isinstance(a, str) and re.fullmatch(r"\\d{4}", a)):\n        errs.append("bad_account_last4")\n    return errs\n\n\n# ---------------------------------------------------------------------------\n# Normalisation & field comparison (lenient on formatting, strict on meaning)\n# ---------------------------------------------------------------------------\ndef _num(v):\n    if v is None:\n        return None\n    if isinstance(v, bool):\n        return None\n    if isinstance(v, (int, float)):\n        return float(v)\n    s = str(v).translate(_DEV)\n    m = re.search(r"\\d[\\d,]*(?:\\.\\d+)?", s)\n    return float(m.group(0).replace(",", "")) if m else None\n\n\ndef _norm_str(v):\n    if v is None:\n        return None\n    s = unicodedata.normalize("NFC", str(v)).translate(_DEV).casefold()\n    s = re.sub(r"\\s+", " ", s).strip()\n    return s or None\n\n\ndef _norm_alnum(v):\n    s = _norm_str(v)\n    return re.sub(r"[^0-9a-z@.]", "", s) if s else None\n\n\ndef norm_field(k, v):\n    if v is None or (isinstance(v, str) and v.strip().lower() in ("", "null", "none", "n/a")):\n        return None\n    if k in ("amount", "balance"):\n        return _num(v)\n    if k == "account_last4":\n        d = re.sub(r"\\D", "", str(v).translate(_DEV))\n        return d[-4:] if d else None\n    if k in ("instrument", "currency"):\n        return _norm_str(v).upper() if _norm_str(v) else None\n    if k in ("type", "status"):\n        return _norm_str(v)\n    if k in ("reference", "counterparty"):\n        return _norm_alnum(v)\n    if k == "date":\n        return _norm_str(v)\n    if k == "is_transaction":\n        if isinstance(v, bool):\n            return v\n        return {"true": True, "false": False}.get(str(v).lower())\n    return _norm_str(v)\n\n\ndef field_equal(k, pred, gold) -> bool:\n    p, g = norm_field(k, pred), norm_field(k, gold)\n    if g is None or p is None:\n        return g is None and p is None\n    if k in ("amount", "balance"):\n        return abs(p - g) < 0.005\n    return p == g\n\n\n# ---------------------------------------------------------------------------\n# Per-example and aggregate scoring\n# ---------------------------------------------------------------------------\ndef score_example(raw_output: str, gold: dict) -> dict:\n    obj = extract_json(raw_output)\n    errs = schema_errors(obj)\n    pred = obj or {}\n    res = {"json_parsed": obj is not None, "schema_valid": obj is not None and not errs,\n           "schema_errors": errs}\n    correct = {}\n    for k in FIELDS:\n        correct[k] = field_equal(k, pred.get(k), gold.get(k))\n    res["field_correct"] = correct\n    res["exact_match"] = obj is not None and all(correct.values())\n    # hallucination: predicted a value where gold is null; omission: the reverse\n    hall = omit = gold_null = gold_val = 0\n    for k in VALUE_FIELDS:\n        g = norm_field(k, gold.get(k))\n        p = norm_field(k, pred.get(k))\n        if g is None:\n            gold_null += 1\n            hall += p is not None\n        else:\n            gold_val += 1\n            omit += p is None\n    res.update(hallucinated=hall, gold_null=gold_null, omitted=omit, gold_valued=gold_val)\n    res["pred_is_txn"] = norm_field("is_transaction", pred.get("is_transaction"))\n    res["gold_is_txn"] = gold["is_transaction"]\n    return res\n\n\ndef aggregate(scored: list[dict]) -> dict:\n    n = len(scored)\n    if n == 0:\n        return {}\n    out = {\n        "n": n,\n        "json_valid": sum(s["json_parsed"] for s in scored) / n,\n        "schema_valid": sum(s["schema_valid"] for s in scored) / n,\n        "exact_match": sum(s["exact_match"] for s in scored) / n,\n    }\n    # micro field accuracy over value fields on transaction examples\n    txn = [s for s in scored if s["gold_is_txn"]]\n    if txn:\n        out["field_acc_txn"] = sum(s["field_correct"][k] for s in txn for k in VALUE_FIELDS) / (\n            len(txn) * len(VALUE_FIELDS))\n        for k in VALUE_FIELDS:\n            out[f"acc_{k}"] = sum(s["field_correct"][k] for s in txn) / len(txn)\n    gn = sum(s["gold_null"] for s in scored)\n    gv = sum(s["gold_valued"] for s in scored)\n    out["hallucination_rate"] = sum(s["hallucinated"] for s in scored) / gn if gn else float("nan")\n    out["omission_rate"] = sum(s["omitted"] for s in scored) / gv if gv else float("nan")\n    # transaction detection (positive class = NOT a transaction, i.e. catching hard negatives)\n    tp = sum(1 for s in scored if s["gold_is_txn"] is False and s["pred_is_txn"] is False)\n    fp = sum(1 for s in scored if s["gold_is_txn"] is True and s["pred_is_txn"] is False)\n    fn = sum(1 for s in scored if s["gold_is_txn"] is False and s["pred_is_txn"] is not False)\n    prec = tp / (tp + fp) if tp + fp else float("nan")\n    rec = tp / (tp + fn) if tp + fn else float("nan")\n    out["nontxn_precision"], out["nontxn_recall"] = prec, rec\n    out["nontxn_f1"] = 2 * prec * rec / (prec + rec) if prec and rec and not math.isnan(prec + rec) else 0.0\n    out["is_txn_acc"] = sum(1 for s in scored if s["pred_is_txn"] == s["gold_is_txn"]) / n\n    return out\n\n\ndef breakdown(rows: list[dict], scored: list[dict], key: str) -> dict:\n    groups = defaultdict(list)\n    for r, s in zip(rows, scored):\n        vals = r.get(key)\n        for v in (vals if isinstance(vals, list) else [vals]):\n            groups[v].append(s)\n    return {g: aggregate(v) for g, v in sorted(groups.items(), key=lambda x: str(x[0]))}\n\n\n# ---------------------------------------------------------------------------\n# Statistics\n# ---------------------------------------------------------------------------\ndef mcnemar_exact(a: list[bool], b: list[bool]) -> dict:\n    """Paired exact McNemar test (a = system A correct, b = system B correct)."""\n    n01 = sum(1 for x, y in zip(a, b) if not x and y)\n    n10 = sum(1 for x, y in zip(a, b) if x and not y)\n    n = n01 + n10\n    if n == 0:\n        return {"b_only": n01, "a_only": n10, "p_value": 1.0}\n    k = min(n01, n10)\n    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n\n    return {"b_only": n01, "a_only": n10, "p_value": min(1.0, 2 * p)}\n\n\ndef bootstrap_ci(values: list[float], n_boot: int = 2000, seed: int = 0) -> tuple[float, float]:\n    rng = random.Random(seed)\n    n = len(values)\n    if n == 0:\n        return (float("nan"), float("nan"))\n    means = sorted(sum(values[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))\n    return means[int(0.025 * n_boot)], means[int(0.975 * n_boot) - 1]\n'
open(f"{WORK}/src/scoring.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/scoring.py', len(CONTENT), 'chars')

In [ ]:
# ---- src/regex_baseline.py ----
CONTENT = '"""A reasonable hand-written rule-based parser: the \'no ML\' baseline.\n\nIt is deliberately decent (not a straw man): it handles the common English formats, the\nusual Hindi/Hinglish keywords, Indian digit grouping and day-first dates.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\n\nMONTHS = {m: i + 1 for i, m in enumerate(\n    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])}\n_DEV = str.maketrans("०१२३४५६७८९", "0123456789")\n\nNEG_HINTS = ["otp", "will be debited", "due on", "cashback", "pre-approved", "offer",\n             "debit honge", "डेबिट किए जाएँगे", "कटेगी", "kategi", "देय है", "due hai",\n             "upcoming", "आगामी", "agli emi", "balance in a/c", "avl bal in your", "शेष राशि", "उपलब्ध शेष"]\nDEBIT_HINTS = ["debited", "debit", "spent", "withdrawn", "sent", "paid", "डेबिट", "खर्च", "निकाले",\n               "निकासी", "भेजे गए", "कटे", "kharch", "nikale", "bheje gaye", "kat gaye", "withdraw",\n               "used for", "चुकाए", "pay hue", "declined", "failed", "विफल", "fail", "nahi ho"]\nCREDIT_HINTS = ["credited", "received", "refund", "reversed", "जमा", "प्राप्त", "रिफ़ंड", "वापस",\n                "credit hue", "mile", "aaya", "wapas", "bheje{@}", "ने आपको", "ne aapko", "salary"]\nAMT_RE = re.compile(\n    r"(?:(?:rs\\.?|inr|₹|रु\\.?|रुपये|usd|eur|gbp|aed|\\$|€|£)\\s*([\\d,]+(?:\\.\\d+)?))|(?:([\\d,]+(?:\\.\\d+)?)\\s*rupaye)",\n    re.I)\n\n\ndef _num(s):\n    return float(s.replace(",", ""))\n\n\ndef _date(t: str):\n    t = t.lower()\n    m = re.search(r"\\b(\\d{4})-(\\d{2})-(\\d{2})\\b", t)\n    if m:\n        return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"\n    m = re.search(r"\\b(\\d{1,2})[-/. ]?(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*[-/. ]?(\\d{2,4})\\b", t)\n    if m:\n        y = int(m.group(3)); y = y + 2000 if y < 100 else y\n        return f"{y:04d}-{MONTHS[m.group(2)]:02d}-{int(m.group(1)):02d}"\n    m = re.search(r"\\b(\\d{1,2})[-/.](\\d{1,2})[-/.](\\d{2,4})\\b", t)\n    if m:\n        y = int(m.group(3)); y = y + 2000 if y < 100 else y\n        return f"{y:04d}-{int(m.group(2)):02d}-{int(m.group(1)):02d}"\n    return None\n\n\ndef parse(sms: str) -> dict:\n    t = sms.translate(_DEV)\n    low = t.lower()\n    out = {k: None for k in ["is_transaction", "type", "amount", "currency", "account_last4",\n                             "instrument", "counterparty", "date", "reference", "balance", "status"]}\n    if any(h in low for h in NEG_HINTS) and not any(h in low for h in ["debited from", "credited"]):\n        out["is_transaction"] = False\n        return out\n    out["is_transaction"] = True\n    is_credit = any(h in low for h in CREDIT_HINTS)\n    out["type"] = "credit" if is_credit else "debit"\n    amts = list(AMT_RE.finditer(t))\n    if amts:\n        m = amts[0]\n        out["amount"] = _num(m.group(1) or m.group(2))\n        head = m.group(0).lower()\n        out["currency"] = ("USD" if ("usd" in head or "$" in head) else "EUR" if ("eur" in head or "€" in head)\n                           else "GBP" if ("gbp" in head or "£" in head) else "AED" if "aed" in head else "INR")\n    m = re.search(r"(?:bal(?:ance)?|शेष(?: राशि)?)\\s*:?\\s*(?:is\\s*)?(?:rs\\.?|inr|₹|रु\\.?|रुपये)?\\s*([\\d,]+(?:\\.\\d+)?)", t, re.I)\n    if m:\n        out["balance"] = _num(m.group(1))\n    m = re.search(r"(?:x+|\\*+|ending\\s|-)(\\d{4})\\b", t, re.I)\n    if m:\n        out["account_last4"] = m.group(1)\n    for inst in ["NEFT", "IMPS", "RTGS", "NACH", "UPI", "ATM"]:\n        if inst.lower() in low:\n            out["instrument"] = inst\n            break\n    if "autopay" in low or "ऑटोपे" in low:\n        out["instrument"] = "AUTOPAY"\n    if "card" in low or "कार्ड" in low:\n        out["instrument"] = "CARD"\n    m = re.search(r"(?:ref(?:erence)?\\.?\\s*(?:no\\.?)?|utr|संदर्भ(?: संख्या)?|रेफ़)\\s*:?\\s*([A-Z0-9]{8,})", t, re.I)\n    if m:\n        out["reference"] = m.group(1)\n    m = re.search(r"(?:to|at|from|by|towards)\\s+(?:vpa\\s+)?([A-Za-z0-9.@ ]+?)(?=\\s+(?:on|via|for|ke|ko|se|par)\\b|[.,;(]|$)", t)\n    if m:\n        out["counterparty"] = m.group(1).strip()\n    out["date"] = _date(t)\n    out["status"] = ("failed" if any(h in low for h in ["fail", "declined", "विफल", "अस्वीकृत", "could not", "nahi ho", "नहीं हो सका", "decline"])\n                     else "reversed" if any(h in low for h in ["reversed", "वापस कर दिए", "wapas aa gaye"])\n                     else "success")\n    return out\n\n\ndef predict(sms: str) -> str:\n    return json.dumps(parse(sms), ensure_ascii=False)\n'
open(f"{WORK}/src/regex_baseline.py", "w", encoding="utf-8").write(CONTENT)
print('wrote src/regex_baseline.py', len(CONTENT), 'chars')

In [ ]:
CHALLENGE_JSONL = '{"id": "challenge-000", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "UPI/P2M/402917365512/BLINKIT -- INR 349.00 Dr. A/c **2231 dt 03-11-26. Avl Bal INR 8,412.09 -Banyan Bank", "gold": {"is_transaction": true, "type": "debit", "amount": 349.0, "currency": "INR", "account_last4": "2231", "instrument": "UPI", "counterparty": "BLINKIT", "date": "2026-11-03", "reference": "402917365512", "balance": 8412.09, "status": "success"}, "tags": ["reordered"], "split": "challenge"}\n{"id": "challenge-001", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "A/c XX0091 Cr by Rs 1,50,000.00 on 01Oct26 NEFT-SUNRISE SOFTWARE LLP. Bal Rs 2,04,551.30", "gold": {"is_transaction": true, "type": "credit", "amount": 150000.0, "currency": "INR", "account_last4": "0091", "instrument": "NEFT", "counterparty": "SUNRISE SOFTWARE LLP", "date": "2026-10-01", "reference": null, "balance": 204551.3, "status": "success"}, "tags": ["cr_dr_abbrev"], "split": "challenge"}\n{"id": "challenge-002", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Rs.12,34,567.89 transferred via RTGS to GANGA TEXTILES LTD from a/c ending 7788 on 15/03/2026. UTR SBINR52026031500421", "gold": {"is_transaction": true, "type": "debit", "amount": 1234567.89, "currency": "INR", "account_last4": "7788", "instrument": "RTGS", "counterparty": "GANGA TEXTILES LTD", "date": "2026-03-15", "reference": "SBINR52026031500421", "balance": null, "status": "success"}, "tags": ["lakh_grouping", "no_currency_symbol_on_balance"], "split": "challenge"}\n{"id": "challenge-003", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Kestrel Bank CC xx4410: Rs.2,999 spent at CROMA on 11-11-26. Avl Lmt Rs.47,001. Total Lmt Rs.50,000.", "gold": {"is_transaction": true, "type": "debit", "amount": 2999.0, "currency": "INR", "account_last4": "4410", "instrument": "CARD", "counterparty": "CROMA", "date": "2026-11-11", "reference": null, "balance": null, "status": "success"}, "tags": ["limit_distractor", "two_amounts"], "split": "challenge"}\n{"id": "challenge-004", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Alert: Card ending 5521 charged EUR 89.90 at BOOKING HOTEL AMSTERDAM on 22 Aug 2026. Not you? Block via app.", "gold": {"is_transaction": true, "type": "debit", "amount": 89.9, "currency": "EUR", "account_last4": "5521", "instrument": "CARD", "counterparty": "BOOKING HOTEL AMSTERDAM", "date": "2026-08-22", "reference": null, "balance": null, "status": "success"}, "tags": ["foreign_currency"], "split": "challenge"}\n{"id": "challenge-005", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Rs 1,250 debited from your a/c XX3390 on 05-09-26 for UPI txn to IRCTC has been reversed. UPI Ref 512309871234.", "gold": {"is_transaction": true, "type": "credit", "amount": 1250.0, "currency": "INR", "account_last4": "3390", "instrument": "UPI", "counterparty": "IRCTC", "date": "2026-09-05", "reference": "512309871234", "balance": null, "status": "reversed"}, "tags": ["reversal"], "split": "challenge"}\n{"id": "challenge-006", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "FAILED: UPI payment of Rs.540 to ZEPTO. Money will be auto-refunded within 48 hrs if deducted. Ref 778812340011", "gold": {"is_transaction": true, "type": "debit", "amount": 540.0, "currency": "INR", "account_last4": null, "instrument": "UPI", "counterparty": "ZEPTO", "date": null, "reference": "778812340011", "balance": null, "status": "failed"}, "tags": ["failed", "no_date"], "split": "challenge"}\n{"id": "challenge-007", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "RS.75000.00CREDITED TO A/C XXXX6120 BY IMPS FROM ANJALI MISHRA ON 18-07-2026.REF 619900123456.AVL BAL RS.81200.00", "gold": {"is_transaction": true, "type": "credit", "amount": 75000.0, "currency": "INR", "account_last4": "6120", "instrument": "IMPS", "counterparty": "ANJALI MISHRA", "date": "2026-07-18", "reference": "619900123456", "balance": 81200.0, "status": "success"}, "tags": ["all_caps", "missing_space"], "split": "challenge"}\n{"id": "challenge-008", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Cash Rs 10,000 withdrawn at ATM MG ROAD from A/c *8845 on 02/02/2026 13:44. Bal Rs 3,215.", "gold": {"is_transaction": true, "type": "debit", "amount": 10000.0, "currency": "INR", "account_last4": "8845", "instrument": "ATM", "counterparty": null, "date": "2026-02-02", "reference": null, "balance": 3215.0, "status": "success"}, "tags": ["atm", "location_text"], "split": "challenge"}\n{"id": "challenge-009", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "NACH debit of Rs.18,450 for HOME LOAN EMI processed on 05-12-2025 from a/c XX0702. Bal Rs.6,010.55", "gold": {"is_transaction": true, "type": "debit", "amount": 18450.0, "currency": "INR", "account_last4": "0702", "instrument": "NACH", "counterparty": "HOME LOAN EMI", "date": "2025-12-05", "reference": null, "balance": 6010.55, "status": "success"}, "tags": ["nach", "emi"], "split": "challenge"}\n{"id": "challenge-010", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "You paid Rs 60 to chaiwala.raju7@neemupi from A/c XX1190 on 09Nov26 via UPI (Ref 600123998877).", "gold": {"is_transaction": true, "type": "debit", "amount": 60.0, "currency": "INR", "account_last4": "1190", "instrument": "UPI", "counterparty": "chaiwala.raju7@neemupi", "date": "2026-11-09", "reference": "600123998877", "balance": null, "status": "success"}, "tags": ["vpa_counterparty"], "split": "challenge"}\n{"id": "challenge-011", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Dear Customer, Rs.4,200.00 has been debited from your account XX5566 towards", "gold": {"is_transaction": true, "type": "debit", "amount": 4200.0, "currency": "INR", "account_last4": "5566", "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": "success"}, "tags": ["truncated"], "split": "challenge"}\n{"id": "challenge-012", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "INR 0.50 credited to a/c x9910 as cashback from BLINKIT on 2026-10-30.", "gold": {"is_transaction": true, "type": "credit", "amount": 0.5, "currency": "INR", "account_last4": "9910", "instrument": null, "counterparty": "BLINKIT", "date": "2026-10-30", "reference": null, "balance": null, "status": "success"}, "tags": ["decimal_no_grouping", "paise"], "split": "challenge"}\n{"id": "challenge-013", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Do not share. 482913 is OTP for txn of INR 12,500.00 at AMAZON on card XX4410. -Kestrel Bank", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "otp_with_amount"], "split": "challenge"}\n{"id": "challenge-014", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Rs 499 will be auto-debited on 20-11-26 for your NETFLIX mandate from a/c XX1190. Ensure sufficient balance.", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "future_debit"], "split": "challenge"}\n{"id": "challenge-015", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Statement for card xx4410: Total due Rs.12,840, Min due Rs.642, pay by 25/11/2026.", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "bill_due"], "split": "challenge"}\n{"id": "challenge-016", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Your available balance in A/c XX5566 is Rs. 9,876.54 as of 14-11-2026 10:00.", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "balance_only"], "split": "challenge"}\n{"id": "challenge-017", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "PRIYA SHARMA has requested Rs.1,500 from you on UPI. Approve only if you know the requester.", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "request_money"], "split": "challenge"}\n{"id": "challenge-018", "family": "challenge", "lang": "en", "template_id": "handwritten", "sms": "Flat Rs 250 off on your first DMART order above Rs 1,500! Use code DM250. Valid till 30 Nov.", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "promo"], "split": "challenge"}\n{"id": "challenge-019", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "बनयान बैंक: UPI रेफ़ 300198765432 | ₹1,299.00 खाता XX4321 से डेबिट | प्राप्तकर्ता MYNTRA | 12.11.2026", "gold": {"is_transaction": true, "type": "debit", "amount": 1299.0, "currency": "INR", "account_last4": "4321", "instrument": "UPI", "counterparty": "MYNTRA", "date": "2026-11-12", "reference": "300198765432", "balance": null, "status": "success"}, "tags": ["reordered"], "split": "challenge"}\n{"id": "challenge-020", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "आपके खाते XX७७१२ में ₹२५,०००.०० NEFT द्वारा VEDA HOSPITALS से जमा हुए, दिनांक ०१-११-२०२६। शेष ₹३१,४०२.१०।", "gold": {"is_transaction": true, "type": "credit", "amount": 25000.0, "currency": "INR", "account_last4": "7712", "instrument": "NEFT", "counterparty": "VEDA HOSPITALS", "date": "2026-11-01", "reference": null, "balance": 31402.1, "status": "success"}, "tags": ["dev_digits"], "split": "challenge"}\n{"id": "challenge-021", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "लेनदेन विफल: ZOMATO को ₹420 का UPI भुगतान नहीं हुआ। कटी हुई राशि 2 दिन में वापस आ जाएगी। रेफ़ 556677889900", "gold": {"is_transaction": true, "type": "debit", "amount": 420.0, "currency": "INR", "account_last4": null, "instrument": "UPI", "counterparty": "ZOMATO", "date": null, "reference": "556677889900", "balance": null, "status": "failed"}, "tags": ["failed"], "split": "challenge"}\n{"id": "challenge-022", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "क्रेडिट कार्ड XX4410 से AMAZON पर ₹5,600 का भुगतान हुआ, 03-10-26। बची हुई लिमिट ₹44,400।", "gold": {"is_transaction": true, "type": "debit", "amount": 5600.0, "currency": "INR", "account_last4": "4410", "instrument": "CARD", "counterparty": "AMAZON", "date": "2026-10-03", "reference": null, "balance": null, "status": "success"}, "tags": ["limit_distractor"], "split": "challenge"}\n{"id": "challenge-023", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "एटीएम से ₹2,000 की निकासी, खाता *8845, 07/06/2026। उपलब्ध शेष ₹1,215।", "gold": {"is_transaction": true, "type": "debit", "amount": 2000.0, "currency": "INR", "account_last4": "8845", "instrument": "ATM", "counterparty": null, "date": "2026-06-07", "reference": null, "balance": 1215.0, "status": "success"}, "tags": ["atm"], "split": "challenge"}\n{"id": "challenge-024", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "UPI लेनदेन (रेफ़ 401122334455) में कटे ₹999 आपके खाते XX3390 में वापस जमा कर दिए गए हैं।", "gold": {"is_transaction": true, "type": "credit", "amount": 999.0, "currency": "INR", "account_last4": "3390", "instrument": "UPI", "counterparty": null, "date": null, "reference": "401122334455", "balance": null, "status": "reversed"}, "tags": ["reversal"], "split": "challenge"}\n{"id": "challenge-025", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "SURESH YADAV ने आपके खाते XX2208 में IMPS से ₹3,500 भेजे हैं। तारीख 21-08-2026।", "gold": {"is_transaction": true, "type": "credit", "amount": 3500.0, "currency": "INR", "account_last4": "2208", "instrument": "IMPS", "counterparty": "SURESH YADAV", "date": "2026-08-21", "reference": null, "balance": null, "status": "success"}, "tags": ["counterparty_person"], "split": "challenge"}\n{"id": "challenge-026", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "₹8,999 के भुगतान के लिए आपका OTP 771204 है। किसी से साझा न करें। - मॉनसून बैंक", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "otp_with_amount"], "split": "challenge"}\n{"id": "challenge-027", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "आपकी LIC PREMIUM की ₹12,000 की किस्त 28-11-2026 को खाता XX0702 से कटेगी।", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "future_debit"], "split": "challenge"}\n{"id": "challenge-028", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "प्रिय ग्राहक, आपके खाते XX5566 का शेष ₹४,५६७.८९ है।", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "balance_only"], "split": "challenge"}\n{"id": "challenge-029", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "त्योहारी ऑफ़र! ₹10,000 से ज़्यादा की खरीदारी पर ₹1,000 का कैशबैक। ऑफ़र 31-10-2026 तक।", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "promo"], "split": "challenge"}\n{"id": "challenge-030", "family": "challenge", "lang": "hi", "template_id": "handwritten", "sms": "आपके खाते XX1190 से ₹15,000 डेबिट किए गए और", "gold": {"is_transaction": true, "type": "debit", "amount": 15000.0, "currency": "INR", "account_last4": "1190", "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": "success"}, "tags": ["truncated"], "split": "challenge"}\n{"id": "challenge-031", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "Banyan Bank alert - ZOMATO ko Rs 312.50 UPI se pay hua | A/c xx4321 | 13-11-26 | Ref 300554433221", "gold": {"is_transaction": true, "type": "debit", "amount": 312.5, "currency": "INR", "account_last4": "4321", "instrument": "UPI", "counterparty": "ZOMATO", "date": "2026-11-13", "reference": "300554433221", "balance": null, "status": "success"}, "tags": ["reordered"], "split": "challenge"}\n{"id": "challenge-032", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "Badhai ho! Salary Rs 1,12,500 aapke A/c XX0091 mein NEFT se credit hui (NORTHSTAR LOGISTICS), 31-10-2026. Bal Rs 1,48,220", "gold": {"is_transaction": true, "type": "credit", "amount": 112500.0, "currency": "INR", "account_last4": "0091", "instrument": "NEFT", "counterparty": "NORTHSTAR LOGISTICS", "date": "2026-10-31", "reference": null, "balance": 148220.0, "status": "success"}, "tags": ["salary", "lakh_grouping"], "split": "challenge"}\n{"id": "challenge-033", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "a/c xx8845 se rs.500debit hue atm se 09-09-26 ko. bal rs.715", "gold": {"is_transaction": true, "type": "debit", "amount": 500.0, "currency": "INR", "account_last4": "8845", "instrument": "ATM", "counterparty": null, "date": "2026-09-09", "reference": null, "balance": 715.0, "status": "success"}, "tags": ["lowercase", "missing_space"], "split": "challenge"}\n{"id": "challenge-034", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "Payment fail! Rs 2,100 OLA ko nahi gaya. Paisa kata ho to 3 din mein wapas. UPI Ref 990011223344", "gold": {"is_transaction": true, "type": "debit", "amount": 2100.0, "currency": "INR", "account_last4": null, "instrument": "UPI", "counterparty": "OLA", "date": null, "reference": "990011223344", "balance": null, "status": "failed"}, "tags": ["failed"], "split": "challenge"}\n{"id": "challenge-035", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "Card XX5521 se USD 12.99 ka payment GOOGLE ONE ko hua, 01-11-2026.", "gold": {"is_transaction": true, "type": "debit", "amount": 12.99, "currency": "USD", "account_last4": "5521", "instrument": "CARD", "counterparty": "GOOGLE ONE", "date": "2026-11-01", "reference": null, "balance": null, "status": "success"}, "tags": ["foreign_currency"], "split": "challenge"}\n{"id": "challenge-036", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "Refund aa gaya! FLIPKART se Rs.1,849 aapke A/c XX2208 mein credit, 06-11-26.", "gold": {"is_transaction": true, "type": "credit", "amount": 1849.0, "currency": "INR", "account_last4": "2208", "instrument": null, "counterparty": "FLIPKART", "date": "2026-11-06", "reference": null, "balance": null, "status": "success"}, "tags": ["refund"], "split": "challenge"}\n{"id": "challenge-037", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "SPOTIFY ke Rs 119 AutoPay se kat gaye A/c XX1190 se, 10-11-2026. Ref AP77120934", "gold": {"is_transaction": true, "type": "debit", "amount": 119.0, "currency": "INR", "account_last4": "1190", "instrument": "AUTOPAY", "counterparty": "SPOTIFY", "date": "2026-11-10", "reference": "AP77120934", "balance": null, "status": "success"}, "tags": ["autopay"], "split": "challenge"}\n{"id": "challenge-038", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "Rs 4,500 ke txn ke liye OTP 339201 hai. Kisi ko mat batana. -Neem Bank", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "otp_with_amount"], "split": "challenge"}\n{"id": "challenge-039", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "ROHIT DAS ne aapse Rs 800 maange hain UPI par. Jaante ho tabhi approve karein.", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "request_money"], "split": "challenge"}\n{"id": "challenge-040", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "Kal 15-11-26 ko aapke A/c XX0702 se Rs 2,300 ki EMI kategi. Balance maintain rakhein.", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "future_debit"], "split": "challenge"}\n{"id": "challenge-041", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "JIO FIBER ka Rs 1,178 ka bill 18-11-2026 tak bharna hai. Abhi pay karein.", "gold": {"is_transaction": false, "type": null, "amount": null, "currency": null, "account_last4": null, "instrument": null, "counterparty": null, "date": null, "reference": null, "balance": null, "status": null}, "tags": ["hard_negative", "bill_due"], "split": "challenge"}\n{"id": "challenge-042", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "आपके A/c XX4321 se Rs 650 debit hue SWIGGY ko UPI se, 14-11-26. Ref 300777888999", "gold": {"is_transaction": true, "type": "debit", "amount": 650.0, "currency": "INR", "account_last4": "4321", "instrument": "UPI", "counterparty": "SWIGGY", "date": "2026-11-14", "reference": "300777888999", "balance": null, "status": "success"}, "tags": ["code_mixed_script"], "split": "challenge"}\n{"id": "challenge-043", "family": "challenge", "lang": "hinglish", "template_id": "handwritten", "sms": "Rs 9,999 aapke Card XX4410 se CROMA par", "gold": {"is_transaction": true, "type": "debit", "amount": 9999.0, "currency": "INR", "account_last4": "4410", "instrument": "CARD", "counterparty": "CROMA", "date": null, "reference": null, "balance": null, "status": "success"}, "tags": ["truncated"], "split": "challenge"}\n'
open(f"{WORK}/data/challenge_set.jsonl", "w", encoding="utf-8").write(CHALLENGE_JSONL)
sys.path.insert(0, f"{WORK}/src")
import importlib, smsgen, scoring as S, regex_baseline as RB
for m in (smsgen, S, RB): importlib.reload(m)
from smsgen import build_dataset, gold_to_str, build_messages, SYSTEM_PROMPT, FIELDS
print('challenge rows:', CHALLENGE_JSONL.count(chr(10)))

In [ ]:
# ---- article/abstract templates and demo Space files ----
ARTICLE_TEMPLATE = '# Teaching a 3B model to read Indian bank SMS, in English, Hindi and Hinglish, without making things up\n\n*Lexsi AI Safety & Alignment Hackathon · Track 1 (Fine-tuning & Data Curation) · Problem 5 (Structured output & extraction), also Problem 2 (replacing a closed-model API) and Problem 4 (multilingual & code-mixed)*\n\n**Model:** `{{BASE_MODEL}}` + LoRA · **Toolkits:** CuratorKIT → AlignTune → AuditKIT\n**Links:** [LoRA adapter](https://huggingface.co/{{MODEL_REPO}}) · [GGUF](https://huggingface.co/{{GGUF_REPO}}) · [Dataset, predictions & logs](https://huggingface.co/datasets/{{DATA_REPO}}) · [Demo](https://huggingface.co/spaces/{{SPACE_REPO}}) · GitHub: `<add your repo link>`\n\n---\n\n## TL;DR\n\nExpense trackers, lending apps and personal-finance tools in India read billions of transaction SMS a year. Many send each message to a closed LLM API. We fine-tuned **Tiny Aya Global (3.35B)** to turn one SMS into one schema-valid JSON object (11 fields), in **English, Hindi (Devanagari) and Hinglish**. It also recognises when a message mentions money but is **not** a transaction: OTPs, offers, bill reminders, future debits and money requests.\n\n- On message formats it **never saw in training**, the fine-tuned model reaches **{{FT_EM_OOD}}%** exact match (all 11 fields right), vs **{{FS_EM_OOD}}%** for few-shot prompting of the same model.\n- On a **hand-written challenge set** of realistic, messy SMS: **{{FT_EM_CH}}%** exact match vs {{FS_EM_CH}}% (few-shot), {{ZS_EM_CH}}% (zero-shot) and {{RX_EM_CH}}% (a hand-written regex parser).\n- **Hallucinated fields** on the challenge set: **{{FT_HALL_CH}}%** vs {{FS_HALL_CH}}% for few-shot.\n- **Ablation:** the same fine-tune *without* our robustness data (hard negatives, noise, missing fields) gets {{ABL_EM_CH}}% exact match and {{ABL_NT_CH}}% non-transaction F1 on the challenge set (vs {{FT_NT_CH}}%).\n- **Cost at {{MONTHLY_SMS}} SMS/month:** **${{SELF_MONTH}}** self-hosted on a T4 ({{GPU_HOURS}} GPU-hours; ${{ALWAYS_ON_MONTH}} for an always-on GPU), vs {{API_MONTH}} per month at closed-API list prices. Fine-tuning itself cost about ${{TRAIN_COST}}.\n\n## 1. Why this problem\n\nA bank SMS looks simple until you meet real ones. *"Rs.2,999 spent at CROMA. Avl Lmt Rs.47,001"* has two amounts, and only one is the transaction; the other is a credit limit, not a balance. *"482913 is OTP for txn of INR 12,500 at AMAZON"* has an amount, a merchant and a card, but no money has moved. *"आपके खाते XX७७१२ में ₹२५,०००.०० जमा हुए"* uses Devanagari digits. Hinglish users type *"rs.500debit hue atm se"*.\n\nFor downstream finance apps, **a wrong value is worse than a missing one**: an invented balance or a phantom transaction corrupts a budget silently. So we optimise and report not just accuracy, but **hallucination**: how often the model fills a field that the SMS does not contain.\n\n## 2. Why this task, this model and this data\n\n**Task.** SMS-to-JSON extraction is a high-volume, repetitive, well-defined job. That is exactly the kind of workload that is usually sent to a closed API and should not need one (Problem 2). It is checkable field by field, so improvement can be measured without an LLM judge (Problem 5). And in India it is inherently multilingual and code-mixed (Problem 4).\n\n**Model: Tiny Aya Global (3.35B).**\n- **Language balance.** It is the Tiny Aya variant tuned for balanced performance across languages and regions. Our inputs mix English, Hindi and romanised Hindi in the same message, so a balanced variant is the safer default. *Tiny Aya Fire* (South-Asia specialised) is a natural follow-up comparison.\n- **Size fits the budget.** At 3.35B parameters, LoRA fine-tuning and batched inference both fit on one free 16 GB T4, and an 8-bit GGUF runs on a 2-vCPU CPU. Aya Expanse 8B/32B would be more capable, but they cannot be fine-tuned on free hardware or served cheaply on-premise, and cheap on-premise serving is the point of the ROI argument.\n- **Small output space.** The output is short and structured (~80 tokens), so model size matters less than format discipline. Fine-tuning targets format discipline directly.\n\n**Data: synthetic, with exact labels.** Real bank SMS contain account numbers, balances and names, so they cannot be collected or published responsibly. Synthetic generation gives (a) **exact gold labels**, (b) **full control over edge cases**, and (c) a **publishable dataset**. The risk of synthetic data is overfitting to templates. We counter it with a template-level held-out split and a hand-written challenge set.\n\n## 3. Data: exact labels by construction\n\nWe wrote a generator (`smsgen.py`) with **14 message families × 3 languages**: UPI debit/credit, card spend (incl. foreign currency), ATM, NEFT/IMPS/RTGS in and out, refunds & reversals, failed transactions, EMI/AutoPay/NACH, and **5 hard-negative families** (OTP, promo, bill due, scheduled debit, balance info).\n\nKey design choices:\n\n1. **Segment templates → exact gold.** Each template is split into segments. Optional segments are randomly dropped and messages are sometimes truncated. A field is labelled **only if its text survived**, so the gold never asks the model to know something that is not in the message. This is what makes the hallucination metric meaningful.\n2. **Template-level held-out split.** For every (family, language), one template is never used for training. `test_ood` contains only those formats, so it measures **generalisation to unseen message formats**, not memorisation of templates.\n3. **Edge cases as first-class data:** Indian digit grouping (₹1,23,456.50), 8 date formats (day-first), masked accounts (`XX1234`, `*1234`, `ending 1234`), credit-limit distractors, promo tails with extra amounts, ALL CAPS / lowercase, missing spaces (`Rs.450debited`), Devanagari numerals, sender headers and footers.\n4. **Responsible synthetic data:** all bank names are fictional and messages contain no links or phone numbers, so the dataset can\'t be confused with, or reused as, real bank messages. No real customer data was used.\n5. **Hand-written challenge set ({{N_CHALLENGE}} SMS):** written and labelled by hand in styles the generator never produces (reordered fields, pipe-separated alerts, "Cr/Dr" abbreviations, money requests, cashback credits, mixed scripts).\n\n| Split | Size | Purpose |\n|---|---|---|\n| train (after CuratorKIT) | {{N_TRAIN}} | fine-tuning |\n| dev | {{N_DEV}} | prompt-format choice only |\n| test_id | {{N_TEST_ID}} | seen formats, new values |\n| test_ood | {{N_TEST_OOD}} | **unseen formats** |\n| challenge | {{N_CHALLENGE}} | **hand-written, messy** |\n\n**CuratorKIT.** The training split is exported in Alpaca format and run through CuratorKIT (cleaning, exact de-duplication, provenance: manifest, checksums, `lexsi_provenance.json`). We then add our own **quality gate**: every curated row\'s output must still parse to JSON identical to its gold label, otherwise it is dropped. We also ran MinHash near-duplicate detection and report its count, but did not apply it: templated SMS are near-duplicates *by design* yet carry different labels, so near-dup removal would delete exactly the minimal pairs the model needs. Curation report: `{{CURATION}}`\n\n## 4. Systems compared\n\n| System | What it is |\n|---|---|\n| `regex` | A careful hand-written rule parser (English + Hindi/Hinglish keywords, Indian grouping, day-first dates). A strong non-ML baseline, not a straw man. |\n| `zero_shot` | Tiny Aya Global with the schema instructions only |\n| `few_shot` | Same + 3 fixed exemplars from train (EN, HI, Hinglish negative) |\n| `ft_no_robust_data` | LoRA fine-tune on the same number of **clean** examples: no hard negatives, no noise, no missing fields (ablation) |\n| `fine_tuned` | **Ours:** LoRA fine-tune on the curated robustness data |\n\nFine-tuning: **AlignTune** `create_sft_trainer` (TRL backend; path used: `{{TRAIN_PATH}}`), LoRA r={{LORA_R}}, α={{LORA_ALPHA}}, lr={{LR}}, {{EPOCHS}} epoch, {{TRAIN_MIN}} min on a single free T4. Greedy decoding. The prompt format (`{{FT_FORMAT}}`) was selected on dev only.\n\n**Evaluation.** Every system on every split was run through **AuditKIT** (run records and fingerprints are in the dataset repo). Training logs and loss curves are in the model repo under `logs/`.\n\n### Metrics, and why each one\n\n| Metric | Definition | Why it matters for this use case |\n|---|---|---|\n| **Exact match** | all 11 fields correct | An app consumes the whole record; one wrong field (e.g. debit vs credit) breaks the budget. This is the headline metric. |\n| **Field accuracy** | per-field correctness on transaction SMS | Shows *where* errors are: amount vs counterparty vs date. |\n| **Hallucination rate ↓** | non-null prediction where gold is null | In finance a made-up balance or reference is worse than a blank; this is the reliability metric. |\n| **Omission rate ↓** | null prediction where gold has a value | The other side of the trade-off: a model that always says null would have zero hallucination. |\n| **Non-transaction F1** | catching OTPs, offers, reminders, future debits | A phantom transaction is the most damaging error for downstream apps. |\n| **Schema validity** | parseable JSON, right keys and types | The integration contract: invalid output needs retries or a human. |\n\nMatching is lenient on formatting (case, whitespace, `1,23,456` vs `123456`, Devanagari digits) and strict on meaning. Significance uses a paired exact McNemar test; intervals use bootstrap resampling.\n\n## 5. Results\n\n### Seen formats (test_id)\n{{TABLE_TEST_ID}}\n\n### Unseen formats (test_ood)\n{{TABLE_TEST_OOD}}\n\n### Hand-written challenge set\n{{TABLE_CHALLENGE}}\n\n### Is the improvement real? (paired exact McNemar test on exact match)\n{{SIG_TABLE}}\n\n### By language: exact match on unseen formats\n{{LANG_TABLE_OOD}}\n\n### By language: exact match on the challenge set\n{{LANG_TABLE_CHALLENGE}}\n\n### Edge cases: exact match on the challenge set\n{{TAG_TABLE_CHALLENGE}}\n\n![Exact match](https://huggingface.co/datasets/{{DATA_REPO}}/resolve/main/figures/exact_match.png)\n![Hallucination](https://huggingface.co/datasets/{{DATA_REPO}}/resolve/main/figures/hallucination.png)\n![Non-transaction F1](https://huggingface.co/datasets/{{DATA_REPO}}/resolve/main/figures/nontxn_f1.png)\n\n## 6. What we learned\n\n> *[EDIT after the run: 3–5 short bullet points in your own words. Prompts to answer from the tables above:]*\n> - *How large is the gap between seen (test_id) and unseen formats (test_ood)? A small gap means the model learned the task, not the templates.*\n> - *Did prompting alone produce valid JSON reliably? (zero-shot schema validity on test_ood: {{ZS_SCHEMA_OOD}}% vs fine-tuned {{FT_SCHEMA_OOD}}%)*\n> - *What did the robustness data buy? Compare `ft_no_robust_data` vs `fine_tuned` on non-transaction F1 and hallucination.*\n> - *Which language is hardest, and which edge case still fails most?*\n\nMost frequent remaining errors (fine-tuned, unseen + challenge): {{WRONG_FIELDS}}. Every error, with gold and prediction, is in `results/error_analysis_fine_tuned.csv`.\n\n## 7. What this means in practice\n\n- **Drop-in for an API call:** same input (one SMS), same output (one JSON object); the schema-validity rate shows how often a retry would be needed.\n- **Safer by default:** a lower hallucination rate and higher non-transaction F1 mean fewer phantom transactions and invented balances in users\' budgets.\n- **Works for how Indians actually text:** Hindi script, Hinglish, Devanagari digits and Indian number grouping are supported natively, with no transliteration step.\n- **Private:** runs on-premise or on a CPU, so no financial SMS leaves the company\'s infrastructure.\n\n## 8. ROI: replacing a closed-model API\n\n{{ROI_TABLE}}\n\nMethod: realistic volume ({{MONTHLY_SMS}} SMS/month) ÷ measured throughput ({{TOK_S}} generated tokens/s, batched greedy decoding on a T4) → GPU hours × hourly GPU price. API cost uses list prices and our measured average prompt/answer lengths (counted with Tiny Aya\'s tokenizer, an approximation). GPT-5-family models may also bill hidden reasoning tokens, so API figures are a **lower bound**. The fine-tuned model also runs on **CPU** as an 8-bit GGUF (the demo Space uses 2 free vCPUs), so on-device or on-prem extraction is possible and no SMS ever leaves the user\'s infrastructure. That is a privacy win for financial data.\n\n**Licence note:** Tiny Aya is released under CC-BY-NC. A commercial deployment needs a licence from Cohere; the cost comparison is about compute, not licensing.\n\n## 9. Limitations\n\n- **Synthetic training data.** Real SMS are more diverse than our 111 templates. The hand-written challenge set and the unseen-format split are our guard against overfitting, but they are small ({{N_CHALLENGE}} and {{N_TEST_OOD}} examples). Results on real bank traffic will be lower.\n- **One author wrote both the generator and the challenge set**, so some stylistic bias is unavoidable. Native-speaker review of more Hindi/Hinglish messages would strengthen it.\n- **Romanised Hindi has no standard spelling.** We cover common spellings only.\n- **Schema decisions are opinions:** failed transactions count as transactions with `status: failed`; future debits do not. Other apps may want different rules.\n- **Single seed, single run** (free GPU budget). Confidence intervals come from bootstrap resampling of examples, not from repeated training runs.\n- **Dates are assumed day-first** (Indian convention).\n\n## 10. Reproduce\n\nOpen the notebook on Kaggle (T4, internet on), add an `HF_TOKEN` secret, *Run all*. `QUICK_MODE=True` runs the whole pipeline on tiny subsets in about 10 minutes. All seeds are fixed. Every split, prediction, metric file, CuratorKIT provenance file and AuditKIT run record is in the dataset repo.\n\n*Built with Lexsi Labs\' CuratorKIT, AlignTune and AuditKIT on Cohere Labs\' Tiny Aya Global.*\n'
ABSTRACT_TEMPLATE = "Indian finance apps read billions of bank and UPI transaction SMS a year, often by sending each message to a closed LLM API. We show that a small open model can do this job reliably, cheaply and on-premise, in English, Hindi (Devanagari) and code-mixed Hinglish.\n\nWe fine-tune Cohere Labs' Tiny Aya Global (3.35B) with LoRA, using a pipeline that chains three Lexsi toolkits: CuratorKIT (cleaning, de-duplication, provenance, plus a JSON quality gate), AlignTune (SFT with the TRL backend on one free T4 GPU) and AuditKIT (evaluation harness for every system and split). The model reads one SMS and returns one schema-valid JSON object with 11 fields: transaction flag, type, amount, currency, account, instrument, counterparty, date, reference, balance and status.\n\nTraining data comes from a generator that labels a field only if its text survives generation, which makes hallucination measurable. It covers 14 message families, including five hard-negative families (OTPs, offers, bill reminders, future debits, balance alerts) that mention money but are not transactions. It also covers edge cases: Indian digit grouping, credit-limit distractors, foreign currency, truncation, Devanagari digits, missing spaces and casing noise. Bank names are fictional. To measure generalisation rather than memorisation, one template per family and language is held out entirely (test_ood). We also evaluate on a hand-written challenge set of realistic, messy messages.\n\nOn unseen formats the fine-tuned model reaches {{FT_EM_OOD}}% exact match (all 11 fields correct), against {{FS_EM_OOD}}% for few-shot prompting of the same model. On the hand-written set it reaches {{FT_EM_CH}}%, against {{FS_EM_CH}}% few-shot, {{ZS_EM_CH}}% zero-shot and {{RX_EM_CH}}% for a hand-written regex parser. Its hallucination rate there is {{FT_HALL_CH}}% (few-shot: {{FS_HALL_CH}}%). An ablation trained on the same amount of clean data, without hard negatives or noise, drops to {{ABL_EM_CH}}% exact match and {{ABL_NT_CH}}% non-transaction F1, which isolates the value of the robustness data. Paired McNemar tests and bootstrap intervals are reported.\n\nAt a realistic {{MONTHLY_SMS}} SMS per month, self-hosting on a T4 costs about ${{SELF_MONTH}} ({{GPU_HOURS}} GPU-hours, from measured throughput), versus {{API_MONTH}} at closed-API list prices. An 8-bit GGUF version runs on a free 2-vCPU demo Space, so SMS never need to leave the user's infrastructure. We release the adapter, the GGUF, all data splits, every prediction, the CuratorKIT provenance and the AuditKIT run records, and we discuss limitations: synthetic training data, a small challenge set and a single training seed.\n"
SPACE_APP = '"""Demo Space: Indian bank/UPI SMS -> JSON with fine-tuned Tiny Aya (runs on free CPU).\n\nBackends, in order: GGUF via llama-cpp-python (fast on CPU) -> transformers + LoRA (slow fallback).\nShows the model\'s JSON, schema validation, and the regex baseline side by side.\n"""\nimport json\nimport os\nimport time\n\nimport gradio as gr\n\nfrom regex_baseline import predict as regex_predict\nfrom scoring import extract_json, schema_errors\nfrom smsgen import SYSTEM_PROMPT, build_messages\n\nGGUF_REPO = os.environ.get("GGUF_REPO", "")\nGGUF_FILE = os.environ.get("GGUF_FILE", "")\nLORA_REPO = os.environ.get("LORA_REPO", "")\nBASE_MODEL = os.environ.get("BASE_MODEL", "CohereLabs/tiny-aya-global")\n\nBACKEND, LLM, TOK = None, None, None\n\n\ndef load():\n    global BACKEND, LLM, TOK\n    if BACKEND:\n        return\n    if GGUF_REPO and GGUF_FILE:\n        try:\n            from huggingface_hub import hf_hub_download\n            from llama_cpp import Llama\n            path = hf_hub_download(GGUF_REPO, GGUF_FILE)\n            LLM = Llama(model_path=path, n_ctx=1024, n_threads=os.cpu_count() or 2, verbose=False)\n            BACKEND = "gguf (llama.cpp, CPU)"\n            return\n        except Exception as e:\n            print("GGUF load failed:", e)\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    TOK = AutoTokenizer.from_pretrained(BASE_MODEL)\n    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)\n    LLM = PeftModel.from_pretrained(base, LORA_REPO).eval()\n    BACKEND = "transformers + LoRA (CPU, slow)"\n\n\ndef run_model(sms: str) -> str:\n    load()\n    msgs = build_messages(sms)\n    if BACKEND.startswith("gguf"):\n        out = LLM.create_chat_completion(messages=msgs, temperature=0.0, max_tokens=200)\n        return out["choices"][0]["message"]["content"]\n    import torch\n    prompt = TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n    enc = TOK(prompt, return_tensors="pt", add_special_tokens=False)\n    with torch.no_grad():\n        gen = LLM.generate(**enc, max_new_tokens=200, do_sample=False)\n    return TOK.decode(gen[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)\n\n\ndef extract(sms: str):\n    if not sms or not sms.strip():\n        return "{}", "Paste an SMS first.", "{}"\n    t0 = time.time()\n    raw = run_model(sms.strip())\n    dt = time.time() - t0\n    obj = extract_json(raw)\n    errs = schema_errors(obj)\n    status = (f"✅ valid JSON, schema OK · {dt:.1f}s · {BACKEND}" if obj is not None and not errs\n              else f"⚠️ {\', \'.join(errs)} · {dt:.1f}s · {BACKEND}")\n    pretty = json.dumps(obj, ensure_ascii=False, indent=2) if obj is not None else raw\n    rx = json.dumps(json.loads(regex_predict(sms)), ensure_ascii=False, indent=2)\n    return pretty, status, rx\n\n\nEXAMPLES = [\n    "Rs.450.00 debited from A/c XX1234 on 14-11-26 to SWIGGY via UPI. UPI Ref 401234567890. Avl bal Rs.12,340.50",\n    "आपके खाते XX७७१२ में ₹२५,०००.०० NEFT द्वारा VEDA HOSPITALS से जमा हुए, दिनांक ०१-११-२०२६। शेष ₹३१,४०२.१०।",\n    "Payment fail! Rs 2,100 OLA ko nahi gaya. Paisa kata ho to 3 din mein wapas. UPI Ref 990011223344",\n    "Kestrel Bank CC xx4410: Rs.2,999 spent at CROMA on 11-11-26. Avl Lmt Rs.47,001. Total Lmt Rs.50,000.",\n    "Do not share. 482913 is OTP for txn of INR 12,500.00 at AMAZON on card XX4410. -Kestrel Bank",\n    "Card XX5521 se USD 12.99 ka payment GOOGLE ONE ko hua, 01-11-2026.",\n    "Kal 15-11-26 ko aapke A/c XX0702 se Rs 2,300 ki EMI kategi. Balance maintain rakhein.",\n]\n\nwith gr.Blocks(title="Tiny Aya · SMS → JSON") as demo:\n    gr.Markdown(\n        "# Tiny Aya · Indian bank & UPI SMS → JSON\\n"\n        "Fine-tuned **Tiny Aya Global (3.35B)** with Lexsi **CuratorKIT → AlignTune → AuditKIT**. "\n        "Works in **English, Hindi and Hinglish**, and returns `is_transaction: false` for OTPs, offers, "\n        "reminders and future debits. Runs on a free CPU. All demo data is synthetic, and bank names are fictional.\\n\\n"\n        "⚠️ *Please don\'t paste real messages that contain personal information.*")\n    with gr.Row():\n        sms = gr.Textbox(label="SMS", lines=4, placeholder="Paste a bank / UPI SMS…")\n    btn = gr.Button("Extract", variant="primary")\n    status = gr.Markdown()\n    with gr.Row():\n        out = gr.Code(label="Fine-tuned Tiny Aya", language="json")\n        rx = gr.Code(label="Regex baseline (for comparison)", language="json")\n    gr.Examples(EXAMPLES, inputs=sms)\n    with gr.Accordion("System prompt & schema", open=False):\n        gr.Markdown(f"```\\n{SYSTEM_PROMPT}\\n```")\n    btn.click(extract, inputs=sms, outputs=[out, status, rx])\n\nif __name__ == "__main__":\n    demo.launch()\n'
SPACE_REQS = '--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu\nllama-cpp-python\ngradio\nhuggingface_hub\ntransformers\npeft\ntorch\n'
SPACE_README = '---\ntitle: Tiny Aya SMS to JSON\nemoji: 🧾\ncolorFrom: indigo\ncolorTo: green\nsdk: gradio\napp_file: app.py\npinned: false\nlicense: cc-by-nc-4.0\nshort_description: Indian bank/UPI SMS to JSON in English, Hindi, Hinglish\n---\n\n# Tiny Aya · Indian bank & UPI SMS → JSON\n\nFine-tuned Tiny Aya Global extracts 11 fields (type, amount, currency, account, instrument, counterparty, date, reference, balance, status) from bank and UPI SMS in English, Hindi and Hinglish. It flags non-transactions such as OTPs, offers, reminders and future debits.\n\n- Model (LoRA): https://huggingface.co/__MODEL_REPO__\n- Model (GGUF, used here): https://huggingface.co/__GGUF_REPO__\n- Data, predictions & metrics: https://huggingface.co/datasets/__DATA_REPO__\n\nBuilt for the Lexsi AI Safety & Alignment Hackathon (Track 1, Problem 5) with CuratorKIT, AlignTune and AuditKIT.\n'
print('templates loaded')

## 3. Generate the dataset (exact labels, template-level held-out split)

In [ ]:
from collections import Counter
ds = build_dataset(N_TRAIN, N_DEV, N_TEST_ID, N_TEST_OOD, seed=SEED)
splits = {k: [e.to_json() for e in v] for k, v in ds.items()}
splits["challenge"] = [json.loads(l) for l in open(f"{WORK}/data/challenge_set.jsonl", encoding="utf-8")]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

for k, rows in splits.items():
    write_jsonl(f"{WORK}/data/{k}.jsonl", rows)

# leakage checks
train_sms = {r["sms"] for r in splits["train"]}
for k in ["dev", "test_id", "test_ood", "challenge"]:
    assert not train_sms & {r["sms"] for r in splits[k]}, f"leak into {k}"
assert not {r["template_id"] for r in splits["test_ood"]} & {r["template_id"] for r in splits["train"]}
print("no SMS overlap and no OOD-template overlap with train ✓")

summary = {k: {"n": len(v), "langs": dict(Counter(r["lang"] for r in v)),
               "non_txn": sum(not r["gold"]["is_transaction"] for r in v)} for k, v in splits.items()}
print(json.dumps(summary, indent=1, ensure_ascii=False))
print("edge-case tags in train:", Counter(t for r in splits["train"] for t in r["tags"]).most_common())
for r in random.Random(1).sample(splits["test_ood"], 3):
    print("\n", r["lang"], r["family"], r["tags"], "\n ", r["sms"], "\n ", gold_to_str(r["gold"]))

## 4. CuratorKIT: clean, de-duplicate, quality-gate, provenance
We export the training split in Alpaca format, run CuratorKIT, then **gate** every curated row: its output must still be valid JSON identical to the gold label, or it is dropped.

In [ ]:
def to_alpaca(r):
    return {"instruction": SYSTEM_PROMPT, "input": f"SMS: {r['sms']}", "output": gold_to_str(r["gold"]), "id": r["id"]}

raw_path = f"{WORK}/data/train_alpaca_raw.jsonl"
write_jsonl(raw_path, [to_alpaca(r) for r in splits["train"]])
by_input = {f"SMS: {r['sms']}": r for r in splits["train"]}
by_id = {r["id"]: r for r in splits["train"]}

def make_config(cls, **kw):
    try:
        params = inspect.signature(cls).parameters
        if not any(p.kind == p.VAR_KEYWORD for p in params.values()):
            kw = {k: v for k, v in kw.items() if k in params}
    except (TypeError, ValueError):
        pass
    return cls(**kw)

def load_any(path):
    if path.endswith(".jsonl"):
        return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    data = json.load(open(path, encoding="utf-8"))
    return data if isinstance(data, list) else data.get("data", [])

curation = {"tool": "CuratorKIT", "ran": False, "rows_in": len(splits["train"]),
            "rows_in_by_lang": dict(Counter(r["lang"] for r in splits["train"]))}

def find_exports(out_dir):
    return [f for f in glob.glob(f"{out_dir}/**/*", recursive=True)
            if os.path.isfile(f) and "alpaca" in f.lower() and f.endswith((".json", ".jsonl"))]

def run_curator(clean, dedup="exact"):
    """Run CuratorKIT; map exported rows back to our rows; apply our JSON quality gate."""
    from curatorkit import Curator, CuratorConfig
    out_dir = f"{WORK}/curated/{dedup}_clean{int(bool(clean))}"
    cfg = make_config(CuratorConfig, dataset=raw_path, dedup=dedup, clean=clean,
                      export_formats=["alpaca"], output_dir=out_dir)
    res = Curator(cfg).run()
    try: res.print_summary()
    except Exception: pass
    exported = [e for f in find_exports(out_dir) for e in load_any(f)]
    kept, dropped_gate = [], 0
    for e in exported:
        inp = e.get("input", "")
        src = by_id.get(e.get("id")) or by_input.get(inp) or by_input.get(f"SMS: {inp}")
        ok = src is not None and S.extract_json(e.get("output", "")) is not None \
             and S.score_example(e.get("output", ""), src["gold"])["exact_match"]
        if ok: kept.append(src)
        else: dropped_gate += 1
    # de-duplicate the mapping (exports may repeat a row) while keeping order
    seen, uniq = set(), []
    for r in kept:
        if r["id"] not in seen:
            seen.add(r["id"]); uniq.append(r)
    stats = {"clean": clean, "dedup": dedup, "rows_exported": len(exported), "rows_after_gate": len(uniq),
             "dropped_by_quality_gate": dropped_gate,
             "kept_by_lang": dict(Counter(r["lang"] for r in uniq)),
             "provenance_files": sorted({os.path.basename(p) for p in glob.glob(f"{out_dir}/**/*", recursive=True)
                                         if os.path.basename(p) in ("manifest.json", "lexsi_provenance.json",
                                                                    "checksums.txt", "dataset_card.md")})}
    return uniq, stats

def retention(stats):
    return {l: stats["kept_by_lang"].get(l, 0) / n for l, n in curation["rows_in_by_lang"].items()}

curated_rows = None
if OK_CURATOR:
    try:
        rows1, st1 = run_curator(clean=True)
        curation["runs"] = [st1]
        ret1 = retention(st1)
        print("retention by language with clean=True:", {k: round(v, 2) for k, v in ret1.items()})
        curated_rows, chosen = rows1, st1
        if min(ret1.values()) < 0.8:
            # CuratorKIT's default text cleaning is tuned for English and can drop Devanagari /
            # romanised Hindi rows. Re-run with clean=False (dedup + provenance only).
            print("⚠️ a language lost >20% of its rows with clean=True -> re-running CuratorKIT with clean=False")
            rows2, st2 = run_curator(clean=False)
            curation["runs"].append(st2)
            if len(rows2) > len(rows1):
                curated_rows, chosen = rows2, st2
        # Restore any remaining rows CuratorKIT dropped, provided they pass our own gate
        # (gold JSON valid and exact-duplicate-free); recorded so the report stays honest.
        kept_ids = {r["id"] for r in curated_rows}
        restored = [r for r in splits["train"] if r["id"] not in kept_ids
                    and S.score_example(gold_to_str(r["gold"]), r["gold"])["exact_match"]]
        if restored and min(retention({"kept_by_lang": dict(Counter(r["lang"] for r in curated_rows))}).values()) < 0.8:
            curated_rows = curated_rows + restored
            curation["restored_rows"] = len(restored)
            curation["restored_by_lang"] = dict(Counter(r["lang"] for r in restored))
        curation.update(ran=True, chosen_run=f"clean={chosen['clean']}", rows_final=len(curated_rows),
                        final_by_lang=dict(Counter(r["lang"] for r in curated_rows)))
        try:   # near-duplicate analysis (reported, not applied)
            _, st3 = run_curator(clean=False, dedup="minhash")
            curation["minhash_rows_kept"] = st3["rows_after_gate"]
        except Exception as e:
            curation["minhash_error"] = str(e)[:300]
    except Exception as e:
        traceback.print_exc()
        curation["error"] = str(e)[:500]
        curated_rows = None

if curated_rows is None:
    print("⚠️ CuratorKIT step failed; falling back to exact de-dup in pandas (recorded in the report).")
    seen, curated_rows = set(), []
    for r in splits["train"]:
        if r["sms"] not in seen:
            seen.add(r["sms"]); curated_rows.append(r)
    curation["fallback"] = "pandas exact dedup"
train_rows = curated_rows
write_jsonl(f"{WORK}/data/train_curated.jsonl", train_rows)
json.dump(curation, open(f"{WORK}/results/curation_report.json", "w"), indent=1)
print(json.dumps(curation, indent=1))

## 5. Load Tiny Aya Global and define the systems

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# T4 (compute capability 7.5) reports bf16 as "supported" but only EMULATES it, ~10x slower.
# Use native bf16 only on Ampere+ (capability >= 8.0); otherwise fp16.
BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| capability:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "-",
      "| dtype:", "bfloat16" if BF16 else "float16")
DTYPE = torch.bfloat16 if BF16 else torch.float16
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# One GPU holds the 3.35B model easily; splitting it across Kaggle's 2 GPUs only adds latency.
DEVICE_MAP = {"": 0} if torch.cuda.is_available() else "cpu"

def load_base(dtype=DTYPE):
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype, device_map=DEVICE_MAP)
    m.eval()
    return m

def to_prompt(messages):
    try:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:   # template without a system role: fold it into the first user turn
        sys_msg = messages[0]["content"]
        msgs = [dict(m) for m in messages[1:]]
        msgs[0]["content"] = sys_msg + "\n\n" + msgs[0]["content"]
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def alpaca_prompt(sms):
    return f"### Instruction:\n{SYSTEM_PROMPT}\n\n### Input:\nSMS: {sms}\n\n### Response:\n"

from transformers import StoppingCriteria, StoppingCriteriaList

def json_closed(text):
    """True once the text contains one complete top-level JSON object."""
    start = text.find("{")
    if start < 0:
        return False
    depth, in_str, esc = 0, False, False
    for c in text[start:]:
        if in_str:
            if esc: esc = False
            elif c == "\\": esc = True
            elif c == '"': in_str = False
        elif c == '"': in_str = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return True
    return False

class StopAfterJson(StoppingCriteria):
    """Per-sequence stop as soon as a full JSON object has been generated (saves ~half the decode time)."""
    def __init__(self, prompt_len):
        self.prompt_len = prompt_len
    def __call__(self, input_ids, scores, **kwargs):
        texts = tok.batch_decode(input_ids[:, self.prompt_len:], skip_special_tokens=True)
        return torch.tensor([json_closed(t) for t in texts], device=input_ids.device)

# Tiny Aya's config sets cache_implementation="hybrid" (a static cache). Recent transformers then
# torch.compile the forward pass and RECOMPILE for every new batch shape (~1 min per batch on a T4).
# A dynamic cache with compilation disabled is ~10x faster for our short, varied prompts.
GEN_EXTRA = None
_GEN_CANDIDATES = [dict(cache_implementation="dynamic", disable_compile=True),
                   dict(cache_implementation="dynamic"),
                   dict(disable_compile=True),
                   dict()]

def _generate_fast(mdl, enc, plen, max_new):
    global GEN_EXTRA
    base = dict(max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id,
                stopping_criteria=StoppingCriteriaList([StopAfterJson(plen)]))
    if GEN_EXTRA is not None:
        return mdl.generate(**enc, **base, **GEN_EXTRA)
    last = None
    for extra in _GEN_CANDIDATES:
        try:
            t0 = time.time()
            out = mdl.generate(**enc, **base, **extra)
            GEN_EXTRA = extra
            print(f"generation settings: {extra or 'model defaults'} (first batch {time.time() - t0:.1f}s)")
            return out
        except (TypeError, ValueError) as e:
            last = e
    raise last

GEN_TOKEN_BUDGET = 12000   # max (batch x padded prompt length) per batch: fits a 16 GB T4 with room for the KV cache

@torch.no_grad()
def _run_batch(prompts, mdl, max_new):
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(mdl.device)
    plen = enc["input_ids"].shape[1]
    try:
        out = _generate_fast(mdl, enc, plen, max_new)
    except torch.cuda.OutOfMemoryError:
        del enc
        gc.collect(); torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        half = len(prompts) // 2           # split and retry instead of crashing the run
        return _run_batch(prompts[:half], mdl, max_new) + _run_batch(prompts[half:], mdl, max_new)
    return [tok.decode(out[j, plen:], skip_special_tokens=True) for j in range(len(prompts))]

def generate(prompts, mdl, batch=GEN_BATCH, max_new=MAX_NEW_TOKENS):
    lens = [len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: lens[i])   # length bucketing
    res, cur = [None] * len(prompts), []
    def flush():
        if cur:
            for i, o in zip(cur, _run_batch([prompts[i] for i in cur], mdl, max_new)):
                res[i] = o
            cur.clear()
    for i in order:
        if cur and (len(cur) >= batch or (len(cur) + 1) * (lens[i] + max_new) > GEN_TOKEN_BUDGET):
            flush()
        cur.append(i)
    flush()
    return res

model = load_base()
probe = generate([to_prompt(build_messages("Rs.450.00 debited from A/c XX1234 on 14-11-26 to SWIGGY via UPI. Avl bal Rs.12,340.50"))], model)[0]
print("probe output:", probe[:300])
if S.extract_json(probe) is None:   # empty or garbled output -> fp16 overflow guard
    print("⚠️ empty output in", DTYPE, "-> reloading in float32 (fp16 overflow guard)")
    del model; gc.collect(); torch.cuda.empty_cache()
    DTYPE = torch.float32
    model = load_base(DTYPE)

In [ ]:
# Few-shot exemplars: fixed, drawn from TRAIN only (one per language, incl. one hard negative)
_rng = random.Random(SEED)
def _pick(lang, txn):
    pool = [r for r in train_rows if r["lang"] == lang and r["gold"]["is_transaction"] == txn] \
        or [r for r in splits["train"] if r["lang"] == lang and r["gold"]["is_transaction"] == txn] \
        or [r for r in splits["train"] if r["gold"]["is_transaction"] == txn]
    clean = [r for r in pool if set(r["tags"]) <= {"hard_negative"}]   # no noise in exemplars
    return _rng.choice(clean or pool)
FEW_SHOT = [(r["sms"], gold_to_str(r["gold"])) for r in [_pick("en", True), _pick("hi", True), _pick("hinglish", False)]]

CACHE = {}   # (system, sms) -> raw output
def run_system(name, sms_list, fn):
    todo = [s for s in dict.fromkeys(sms_list) if (name, s) not in CACHE]
    if todo:
        t0 = time.time()
        for s, o in zip(todo, fn(todo)):
            CACHE[(name, s)] = o
        print(f"  {name}: {len(todo)} generations in {time.time() - t0:.0f}s")
    return [CACHE[(name, s)] for s in sms_list]

SYSTEMS = {
    "regex":     lambda xs: [RB.predict(x) for x in xs],
    "zero_shot": lambda xs: generate([to_prompt(build_messages(x)) for x in xs], model),
    "few_shot":  lambda xs: generate([to_prompt(build_messages(x, FEW_SHOT)) for x in xs], model),
}
EVAL_SPLITS = ["test_id", "test_ood", "challenge"]

## 6. AuditKIT evaluation harness
AuditKIT runs every system on every split (its run record, fingerprint and headline are saved). Generations are batched and cached first, so AuditKIT's per-call model function just reads the cache. Our task-specific metrics are computed on the same predictions.

In [ ]:
ak = None
if OK_AUDIT:
    try:
        import auditkit as ak
    except Exception as e:
        print("AuditKIT import failed:", e)

def _jsonable(x):
    try:
        json.dumps(x); return x
    except Exception:
        return str(x)

AUDIT_RUNS = {}
def audit_and_score(system, split):
    rows = splits[split]
    sms = [r["sms"] for r in rows]
    outputs = run_system(system, sms, SYSTEMS[system])
    meta = {"auditkit": False}
    if ak is not None:
        try:
            samples = [ak.Sample(input=r["sms"], target=gold_to_str(r["gold"])) for r in rows]
            model_fn = lambda prompts: [CACHE.get((system, p)) or SYSTEMS[system]([p])[0] for p in prompts]
            result = ak.evaluate(samples, model=model_fn)
            AUDIT_RUNS[(system, split)] = result
            meta = {"auditkit": True, "headline": _jsonable(getattr(result, "headline", None)),
                    "fingerprint": _jsonable(getattr(result, "fingerprint", None))}
            try:
                meta["summary"] = str(result.summary())[:2000]
            except Exception:
                pass
        except Exception as e:
            meta["auditkit_error"] = str(e)[:300]
    scored = [S.score_example(o, r["gold"]) for o, r in zip(outputs, rows)]
    return {"metrics": S.aggregate(scored), "by_lang": S.breakdown(rows, scored, "lang"),
            "by_tag": S.breakdown(rows, scored, "tags"), "scored": scored, "outputs": outputs, "audit": meta}

RESULTS = {}
def evaluate_system(system, split_names=EVAL_SPLITS):
    for sp in split_names:
        RESULTS[(system, sp)] = audit_and_score(system, sp)
        m = RESULTS[(system, sp)]["metrics"]
        print(f"{system:>12} | {sp:<9} | EM {m['exact_match']:.3f} | field acc {m.get('field_acc_txn', float('nan')):.3f}"
              f" | halluc {m['hallucination_rate']:.3f} | non-txn F1 {m['nontxn_f1']:.3f} | schema {m['schema_valid']:.3f}")

for s in ["regex", "zero_shot", "few_shot"]:
    evaluate_system(s)

## 7. Fine-tune with AlignTune (LoRA SFT)
AlignTune's `create_sft_trainer` with the TRL backend. The cell inspects the function signature and passes only arguments it accepts. If AlignTune cannot run in this environment, it falls back to plain TRL + PEFT with the same hyper-parameters, and **the report records which path was used**.

In [ ]:
from datasets import Dataset

def build_train_dataset(rows):
    recs = []
    for r in rows:
        msgs = build_messages(r["sms"]) + [{"role": "assistant", "content": gold_to_str(r["gold"])}]
        prompt = to_prompt(build_messages(r["sms"]))
        recs.append({"messages": msgs, "text": prompt + gold_to_str(r["gold"]) + (tok.eos_token or ""),
                     "instruction": SYSTEM_PROMPT, "input": f"SMS: {r['sms']}", "output": gold_to_str(r["gold"])})
    return Dataset.from_list(recs)

def free_gpu():
    global model
    try: del model
    except NameError: pass
    gc.collect(); torch.cuda.empty_cache()

def find_artifact(out_dir):
    for pat in ["**/adapter_config.json", "**/config.json"]:
        hits = sorted(glob.glob(f"{out_dir}/{pat}", recursive=True), key=os.path.getmtime)
        if hits:
            return os.path.dirname(hits[-1])
    return None

def get_log_history(trainer):
    for obj in (trainer, getattr(trainer, "trainer", None), getattr(trainer, "_trainer", None)):
        st = getattr(obj, "state", None) if obj is not None else None
        if st is not None and getattr(st, "log_history", None):
            return _jsonable(st.log_history)
    return None

def train_aligntune(rows, out_dir):
    from aligntune.core.backend_factory import create_sft_trainer
    data_dir = f"{out_dir}_data"
    ds_full = build_train_dataset(rows)
    variants = {
        "messages": ds_full.select_columns(["messages"]),
        "text": ds_full.select_columns(["text"]),
        "alpaca": ds_full.select_columns(["instruction", "input", "output"]),
    }
    params = inspect.signature(create_sft_trainer).parameters
    explicit = {k for k, p in params.items() if p.kind not in (p.VAR_KEYWORD, p.VAR_POSITIONAL)}
    has_kwargs = any(p.kind == p.VAR_KEYWORD for p in params.values())
    print("create_sft_trainer parameters:", sorted(explicit), "| **kwargs:", has_kwargs)
    core = dict(model_name=BASE_MODEL, backend="trl", num_epochs=EPOCHS, max_steps=-1,
                batch_size=TRAIN_BATCH, learning_rate=LR)
    extra = dict(output_dir=out_dir, max_seq_length=MAX_SEQ_LEN, gradient_accumulation_steps=GRAD_ACCUM,
                 use_peft=True, peft=True, use_lora=True, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
                 lora_dropout=LORA_DROPOUT, gradient_checkpointing=True, seed=SEED,
                 fp16=not BF16, bf16=BF16, logging_steps=10, save_strategy="no", report_to="none")
    last_err = None
    for fmt, d in variants.items():
        path = f"{data_dir}/{fmt}"
        d.to_json(f"{path}.jsonl", force_ascii=False)
        for ds_ref in [f"{path}.jsonl", path]:
            if ds_ref == path:
                d.save_to_disk(path)
            kw = dict(core, dataset_name=ds_ref)
            if fmt == "text":
                extra_fmt = dict(dataset_text_field="text", text_field="text")
            else:
                extra_fmt = {}
            for k, v in {**extra, **extra_fmt}.items():
                if k in explicit:
                    kw[k] = v
            try:
                print(f"AlignTune attempt: format={fmt} dataset={ds_ref} args={sorted(kw)}")
                trainer = create_sft_trainer(**kw)
                trainer.train()
                saved = None
                for obj in (trainer, getattr(trainer, "trainer", None)):
                    m = getattr(obj, "model", None) if obj is not None else None
                    if m is not None and hasattr(m, "save_pretrained"):
                        m.save_pretrained(f"{out_dir}/final"); saved = f"{out_dir}/final"; break
                for meth in ("save_model", "save"):
                    if saved is None and hasattr(trainer, meth):
                        try:
                            getattr(trainer, meth)(f"{out_dir}/final"); saved = f"{out_dir}/final"
                        except Exception:
                            pass
                art = find_artifact(out_dir) or saved
                try:
                    ev = trainer.evaluate()
                except Exception:
                    ev = None
                return {"path": "aligntune", "format": fmt, "dataset_ref": ds_ref, "artifact": art,
                        "args": {k: str(v) for k, v in kw.items()}, "aligntune_eval": _jsonable(ev),
                        "log_history": get_log_history(trainer)}
            except Exception as e:
                last_err = e
                print("  failed:", str(e)[:400])
                gc.collect(); torch.cuda.empty_cache()
                if "out of memory" in str(e).lower():
                    raise RuntimeError("AlignTune ran out of GPU memory (likely full fine-tuning)") from e
    raise RuntimeError(f"AlignTune could not train: {last_err}")

def train_trl_fallback(rows, out_dir):
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE, device_map=DEVICE_MAP)
    cfg_kw = dict(output_dir=out_dir, num_train_epochs=EPOCHS, per_device_train_batch_size=TRAIN_BATCH,
                  gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, lr_scheduler_type="cosine",
                  warmup_ratio=0.05, logging_steps=10, save_strategy="no", report_to="none", seed=SEED,
                  fp16=not BF16 and DTYPE != torch.float32, bf16=BF16, gradient_checkpointing=True)
    sig = inspect.signature(SFTConfig).parameters
    cfg_kw["max_length" if "max_length" in sig else "max_seq_length"] = MAX_SEQ_LEN
    trainer = SFTTrainer(model=base, args=SFTConfig(**cfg_kw),
                         train_dataset=build_train_dataset(rows).select_columns(["messages"]),
                         processing_class=tok,
                         peft_config=LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                                target_modules="all-linear", task_type="CAUSAL_LM"))
    trainer.train()
    trainer.model.save_pretrained(f"{out_dir}/final")
    return {"path": "trl_fallback", "format": "messages", "artifact": f"{out_dir}/final",
            "log_history": get_log_history(trainer)}

def finetune(rows, tag):
    out_dir = f"{WORK}/models/{tag}"
    free_gpu()
    t0 = time.time()
    info = None
    if OK_ALIGN:
        try:
            info = train_aligntune(rows, out_dir)
        except Exception as e:
            traceback.print_exc()
            print("⚠️ AlignTune path failed -> TRL+PEFT fallback (recorded in report)")
    if info is None:
        info = train_trl_fallback(rows, out_dir)
    info["train_minutes"] = round((time.time() - t0) / 60, 1)
    info["n_train"] = len(rows)
    save_train_log(info.pop("log_history", None), tag)
    print(json.dumps(info, indent=1))
    return info

def save_train_log(log, tag):
    """Training logs -> results/ (uploaded to the HF dataset and model repos) + loss curve."""
    json.dump(log or [], open(f"{WORK}/results/train_log_{tag}.json", "w"), indent=1)
    pts = [(e.get("step"), e.get("loss")) for e in (log or []) if isinstance(e, dict) and e.get("loss") is not None]
    if pts:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6, 3.2))
        ax.plot([p[0] for p in pts], [p[1] for p in pts])
        ax.set_xlabel("step"); ax.set_ylabel("training loss"); ax.set_title(f"Training loss ({tag})")
        ax.spines[["top", "right"]].set_visible(False)
        fig.tight_layout(); fig.savefig(f"{WORK}/figures/train_loss_{tag}.png", dpi=150); plt.close(fig)
    print(f"training log: {len(log or [])} entries -> results/train_log_{tag}.json")

def load_finetuned(info):
    from peft import PeftModel
    art = info["artifact"]
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE, device_map=DEVICE_MAP)
    if art and os.path.exists(os.path.join(art, "adapter_config.json")):
        m = PeftModel.from_pretrained(base, art)
    else:
        del base; gc.collect(); torch.cuda.empty_cache()
        m = AutoModelForCausalLM.from_pretrained(art, torch_dtype=DTYPE, device_map=DEVICE_MAP)
    m.eval()
    return m

FT_INFO = finetune(train_rows, "ft_main")
json.dump(FT_INFO, open(f"{WORK}/results/train_info_main.json", "w"), indent=1)

## 8. Evaluate the fine-tuned model
The prompt format (chat template vs Alpaca) is chosen on the **dev** split only, because the trainer may have formatted examples either way. The test splits are never used for any choice.

In [ ]:
model = load_finetuned(FT_INFO)
dev = splits["dev"]
fmt_scores = {}
for fmt, builder in {"chat": lambda x: to_prompt(build_messages(x)), "alpaca": alpaca_prompt}.items():
    outs = generate([builder(r["sms"]) for r in dev], model)
    fmt_scores[fmt] = S.aggregate([S.score_example(o, r["gold"]) for o, r in zip(outs, dev)])["exact_match"]
FT_FORMAT = max(fmt_scores, key=fmt_scores.get)
print("dev exact match by prompt format:", fmt_scores, "-> using", FT_FORMAT)
_build = (lambda x: to_prompt(build_messages(x))) if FT_FORMAT == "chat" else alpaca_prompt
SYSTEMS["fine_tuned"] = lambda xs: generate([_build(x) for x in xs], model)
evaluate_system("fine_tuned")

# throughput for the ROI section (batched greedy decoding on this GPU)
sample = [r["sms"] for r in splits["test_id"][:min(128, len(splits["test_id"]))]]
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.time(); _outs = generate([_build(x) for x in sample], model); dt = time.time() - t0
SMS_PER_SEC = len(sample) / dt
GEN_TOKENS_PER_SEC = sum(len(tok(o, add_special_tokens=False)["input_ids"]) for o in _outs) / dt
print(f"throughput: {SMS_PER_SEC:.2f} SMS/s, {GEN_TOKENS_PER_SEC:.0f} generated tokens/s "
      f"(batch {GEN_BATCH}, greedy) on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 9. Ablation: the same fine-tune without the robustness data
Same size and hyper-parameters, but **no hard negatives, no noise, no dropped fields**. This isolates what the edge-case data contributes.

In [ ]:
if RUN_ABLATION:
    eval_sms = {r["sms"] for k in ["dev", "test_id", "test_ood", "challenge"] for r in splits[k]}
    abl = build_dataset(len(train_rows), 0, 0, 0, seed=SEED + 1, neg_share=0.0, noise_p=0.0,
                        allow_drop=False, exclude_sms=eval_sms)["train"]
    abl_rows = [e.to_json() for e in abl]
    write_jsonl(f"{WORK}/data/train_ablation_clean.jsonl", abl_rows)
    free_gpu()
    ABL_INFO = finetune(abl_rows, "ft_ablation_clean")
    json.dump(ABL_INFO, open(f"{WORK}/results/train_info_ablation.json", "w"), indent=1)
    model_abl = load_finetuned(ABL_INFO)
    SYSTEMS["ft_no_robust_data"] = lambda xs: generate([_build(x) for x in xs], model_abl)
    evaluate_system("ft_no_robust_data")
    del model_abl; gc.collect(); torch.cuda.empty_cache()

## 10. Results tables, significance tests, charts

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

systems = [s for s in ["regex", "zero_shot", "few_shot", "ft_no_robust_data", "fine_tuned"] if (s, "test_id") in RESULTS]
KEYS = [("exact_match", "Exact match"), ("field_acc_txn", "Field acc."), ("hallucination_rate", "Halluc. ↓"),
        ("omission_rate", "Omission ↓"), ("nontxn_f1", "Non-txn F1"), ("schema_valid", "Schema valid")]

def pct(x):
    return "–" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{100 * x:.1f}"

def table(split):
    lines = ["| System | " + " | ".join(n for _, n in KEYS) + " |", "|---|" + "---|" * len(KEYS)]
    for s in systems:
        m = RESULTS[(s, split)]["metrics"]
        lines.append(f"| {s} | " + " | ".join(pct(m.get(k)) for k, _ in KEYS) + " |")
    return "\n".join(lines)

def em_ci(s, split):
    v = [float(x["exact_match"]) for x in RESULTS[(s, split)]["scored"]]
    return S.bootstrap_ci(v, seed=SEED)

TABLES = {sp: table(sp) for sp in EVAL_SPLITS}
for sp in EVAL_SPLITS:
    print(f"\n### {sp} (n={len(splits[sp])})\n{TABLES[sp]}")

# Paired significance: fine-tuned vs each other system (exact match per example)
SIG = {}
for sp in EVAL_SPLITS:
    ft = [x["exact_match"] for x in RESULTS[("fine_tuned", sp)]["scored"]]
    for s in systems:
        if s == "fine_tuned":
            continue
        other = [x["exact_match"] for x in RESULTS[(s, sp)]["scored"]]
        SIG[(s, sp)] = S.mcnemar_exact(other, ft)
sig_lines = ["| Comparison | Split | FT wins | Other wins | McNemar p |", "|---|---|---|---|---|"]
for (s, sp), r in SIG.items():
    sig_lines.append(f"| fine_tuned vs {s} | {sp} | {r['b_only']} | {r['a_only']} | {r['p_value']:.2g} |")
SIG_TABLE = "\n".join(sig_lines)
print("\n" + SIG_TABLE)

# AuditKIT paired comparison (if the installed version exposes it)
AK_COMPARE = {}
if ak is not None:
    for sp in EVAL_SPLITS:
        a, b = AUDIT_RUNS.get(("few_shot", sp)), AUDIT_RUNS.get(("fine_tuned", sp))
        for fn_name in ("compare", "compare_runs"):
            if a is not None and b is not None and hasattr(ak, fn_name):
                try:
                    AK_COMPARE[sp] = str(getattr(ak, fn_name)(a, b).summary())[:1500]; break
                except Exception as e:
                    AK_COMPARE[sp] = f"error: {e}"[:300]
print("AuditKIT compare:", AK_COMPARE if AK_COMPARE else "not available in this AuditKIT version")

# Per-language & per-edge-case tables on the hardest splits
def lang_table(split):
    langs = ["en", "hi", "hinglish"]
    lines = ["| System | " + " | ".join(f"{l} EM" for l in langs) + " |", "|---|" + "---|" * len(langs)]
    for s in systems:
        bl = RESULTS[(s, split)]["by_lang"]
        lines.append(f"| {s} | " + " | ".join(pct(bl.get(l, {}).get("exact_match")) for l in langs) + " |")
    return "\n".join(lines)

def tag_table(split, top=14):
    tags = Counter(t for r in splits[split] for t in r["tags"])
    tags = [t for t, _ in tags.most_common(top)]
    shown = [s for s in ["regex", "few_shot", "ft_no_robust_data", "fine_tuned"] if s in systems]
    lines = ["| Edge case (n) | " + " | ".join(shown) + " |", "|---|" + "---|" * len(shown)]
    for t in tags:
        n = sum(t in r["tags"] for r in splits[split])
        lines.append(f"| {t} ({n}) | " + " | ".join(pct(RESULTS[(s, split)]["by_tag"].get(t, {}).get("exact_match")) for s in shown) + " |")
    return "\n".join(lines)

LANG_TABLES = {sp: lang_table(sp) for sp in EVAL_SPLITS}
TAG_TABLES = {sp: tag_table(sp) for sp in ["test_ood", "challenge"]}
print("\n" + LANG_TABLES["test_ood"] + "\n\n" + TAG_TABLES["challenge"])

# Charts
def bar(metric, title, fname):
    fig, ax = plt.subplots(figsize=(9, 4))
    w = 0.8 / len(systems)
    for i, s in enumerate(systems):
        vals = [100 * (RESULTS[(s, sp)]["metrics"].get(metric) or 0) for sp in EVAL_SPLITS]
        ax.bar([x + i * w for x in range(len(EVAL_SPLITS))], vals, w, label=s)
    ax.set_xticks([x + w * (len(systems) - 1) / 2 for x in range(len(EVAL_SPLITS))])
    ax.set_xticklabels(["seen formats", "unseen formats", "hand-written"])
    ax.set_ylabel("%"); ax.set_title(title); ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(f"{WORK}/figures/{fname}", dpi=160); plt.close(fig)

bar("exact_match", "Exact match (all 11 fields correct)", "exact_match.png")
bar("hallucination_rate", "Hallucinated fields (lower is better)", "hallucination.png")
bar("nontxn_f1", "Catching non-transactions (OTP, offers, reminders): F1", "nontxn_f1.png")
print("charts saved to", f"{WORK}/figures")

## 11. Error analysis
Every fine-tuned failure on the unseen-format and hand-written splits, with the exact fields that went wrong.

In [ ]:
import csv
err_rows = []
for sp in ["test_ood", "challenge"]:
    res = RESULTS[("fine_tuned", sp)]
    for r, sc, out in zip(splits[sp], res["scored"], res["outputs"]):
        if not sc["exact_match"]:
            pred = S.extract_json(out) or {}
            wrong = [k for k, ok in sc["field_correct"].items() if not ok]
            err_rows.append({"split": sp, "id": r["id"], "lang": r["lang"], "tags": ";".join(r["tags"]),
                             "wrong_fields": ";".join(wrong), "sms": r["sms"],
                             "gold": gold_to_str(r["gold"]), "pred": out.strip()[:600]})
with open(f"{WORK}/results/error_analysis_fine_tuned.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(err_rows[0].keys()) if err_rows else ["split"])
    w.writeheader(); w.writerows(err_rows)
WRONG_FIELD_COUNTS = Counter(k for e in err_rows for k in e["wrong_fields"].split(";") if k)
print(len(err_rows), "errors | most common wrong fields:", WRONG_FIELD_COUNTS.most_common(6))
for e in err_rows[:8]:
    print("\n", e["split"], e["lang"], e["tags"], "| wrong:", e["wrong_fields"], "\n ", e["sms"], "\n  gold:", e["gold"], "\n  pred:", e["pred"][:300])

## 12. ROI: self-hosted Tiny Aya vs a closed-model API
Realistic monthly volume × measured throughput (SMS/s and tokens/s) → GPU hours × hourly price, vs per-token API billing. Token counts are measured with each model's own prompt (zero-shot instructions for the API, same as our system prompt). Note: GPT-5-family models may also bill hidden reasoning tokens as output, so API figures are a **lower bound**.

In [ ]:
def n_tokens(text):
    return len(tok(text, add_special_tokens=False)["input_ids"])

sample_rows = splits["test_id"][:200]
avg_in = sum(n_tokens(SYSTEM_PROMPT) + n_tokens("SMS: " + r["sms"]) for r in sample_rows) / len(sample_rows)
avg_out = sum(n_tokens(gold_to_str(r["gold"])) for r in sample_rows) / len(sample_rows)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
gpu_hours = MONTHLY_SMS / (SMS_PER_SEC * 3600)
train_cost = (FT_INFO.get("train_minutes") or 0) / 60 * GPU_USD_PER_HOUR
ROI = {"monthly_sms": MONTHLY_SMS, "avg_input_tokens": round(avg_in, 1), "avg_output_tokens": round(avg_out, 1),
       "self_hosted": {"gpu": gpu_name, "usd_per_gpu_hour": GPU_USD_PER_HOUR,
                       "sms_per_sec": round(SMS_PER_SEC, 2), "generated_tokens_per_sec": round(GEN_TOKENS_PER_SEC, 1),
                       "gpu_hours_per_month": round(gpu_hours, 1),
                       "usd_per_month_on_demand": round(gpu_hours * GPU_USD_PER_HOUR, 2),
                       "usd_per_month_always_on_1gpu": round(730 * GPU_USD_PER_HOUR, 2),
                       "one_time_training_usd": round(train_cost, 2),
                       "usd_per_1M_sms": round(GPU_USD_PER_HOUR / (SMS_PER_SEC * 3600) * 1e6, 2)},
       "api": {}}
for name, (pin, pout) in API_PRICES.items():
    per_sms = (avg_in * pin + avg_out * pout) / 1e6
    ROI["api"][name] = {"usd_per_1M_sms": round(per_sms * 1e6, 2), "usd_per_month": round(per_sms * MONTHLY_SMS, 2),
                        "input_tokens_per_month_M": round(avg_in * MONTHLY_SMS / 1e6, 1),
                        "output_tokens_per_month_M": round(avg_out * MONTHLY_SMS / 1e6, 1)}
json.dump(ROI, open(f"{WORK}/results/roi.json", "w"), indent=1)
print(json.dumps(ROI, indent=1))
sh = ROI["self_hosted"]
ROI_TABLE = (
    f"Scenario: **{MONTHLY_SMS:,} SMS / month** · avg {avg_in:.0f} input + {avg_out:.0f} output tokens per SMS\n\n"
    "| Option | Throughput | GPU hours / month | Cost / month (USD) | Cost / 1M SMS (USD) |\n|---|---|---|---|---|\n"
    f"| Tiny Aya fine-tuned, self-hosted, pay-per-hour ({sh['gpu']} @ ${GPU_USD_PER_HOUR}/h) | {sh['sms_per_sec']} SMS/s · {sh['generated_tokens_per_sec']:.0f} tok/s | {sh['gpu_hours_per_month']:,.1f} | {sh['usd_per_month_on_demand']:,.2f} | {sh['usd_per_1M_sms']:,.2f} |\n"
    f"| Tiny Aya fine-tuned, one always-on GPU (730 h) | same | 730 | {sh['usd_per_month_always_on_1gpu']:,.2f} | – |\n"
    + "\n".join(f"| {k} API (list price; excludes reasoning tokens) | – | – | {v['usd_per_month']:,.2f} | {v['usd_per_1M_sms']:,.2f} |" for k, v in ROI["api"].items())
    + f"\n\nOne-time fine-tuning cost: {FT_INFO.get('train_minutes')} GPU-minutes ≈ ${sh['one_time_training_usd']:.2f}."
)
print(ROI_TABLE)

## 13. Save everything, then publish to the Hugging Face Hub
Model repos (LoRA adapter, GGUF for CPU), dataset repo (all splits, curated train set, every prediction, metrics), two collections, and the demo Space.

In [ ]:
# ---- save metrics & predictions locally ----
all_metrics = {f"{s}/{sp}": {"metrics": v["metrics"], "by_lang": v["by_lang"], "audit": v["audit"]}
               for (s, sp), v in RESULTS.items()}
json.dump(all_metrics, open(f"{WORK}/results/metrics.json", "w"), indent=1, ensure_ascii=False, default=str)
for (s, sp), v in RESULTS.items():
    write_jsonl(f"{WORK}/results/predictions_{s}_{sp}.jsonl",
                [{"id": r["id"], "sms": r["sms"], "gold": r["gold"], "output": o, "exact_match": sc["exact_match"]}
                 for r, o, sc in zip(splits[sp], v["outputs"], v["scored"])])
json.dump({"roi": ROI, "significance": {f"{a}/{b}": v for (a, b), v in SIG.items()},
           "ft_prompt_format": FT_FORMAT, "dev_format_scores": fmt_scores, "train_info": FT_INFO,
           "auditkit_compare": AK_COMPARE},
          open(f"{WORK}/results/summary.json", "w"), indent=1, default=str)
print("saved results to", f"{WORK}/results")

In [ ]:
MODEL_REPO = f"{HF_USERNAME}/{PROJECT}-lora"
GGUF_REPO  = f"{HF_USERNAME}/{PROJECT}-gguf"
DATA_REPO  = f"{HF_USERNAME}/{PROJECT}-data"
SPACE_REPO = f"{HF_USERNAME}/{PROJECT}-demo"
ABL_REPO   = f"{HF_USERNAME}/{PROJECT}-lora-ablation-clean"

def headline(split, s="fine_tuned"):
    return RESULTS[(s, split)]["metrics"]

MODEL_CARD = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: cc-by-nc-4.0
language: [en, hi]
tags: [lexsi-hackathon, structured-output, information-extraction, upi, banking, hinglish, lora, aligntune]
---
# Tiny Aya Global, fine-tuned for Indian bank/UPI SMS → JSON

LoRA adapter for `{BASE_MODEL}` trained with **AlignTune** on data curated with **CuratorKIT** and evaluated with **AuditKIT**.
Input: one SMS in English, Hindi or Hinglish. Output: one JSON object with 11 fields (see the dataset card).

| Split | Exact match | Field accuracy | Hallucination ↓ | Non-txn F1 |
|---|---|---|---|---|
""" + "\n".join(f"| {sp} | {pct(headline(sp)['exact_match'])} | {pct(headline(sp).get('field_acc_txn'))} | {pct(headline(sp)['hallucination_rate'])} | {pct(headline(sp)['nontxn_f1'])} |" for sp in EVAL_SPLITS) + f"""

Training path: `{FT_INFO['path']}` · prompt format: `{FT_FORMAT}` · {FT_INFO['n_train']} curated examples · LoRA r={LORA_R}, α={LORA_ALPHA}, lr={LR}, {EPOCHS} epoch(s).

**Licence:** the base model is CC-BY-NC (non-commercial). Commercial deployment needs a licence from Cohere.
**Data:** fully synthetic; bank names are fictional; no real customer data.
"""

In [ ]:
if PUSH_TO_HUB:
    # ---- model: LoRA adapter (+ card) ----
    api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
    art = FT_INFO["artifact"]
    open(os.path.join(art, "README.md"), "w").write(MODEL_CARD)
    api.upload_folder(folder_path=art, repo_id=MODEL_REPO, repo_type="model")
    for f in glob.glob(f"{WORK}/results/train_log_*.json") + glob.glob(f"{WORK}/results/train_info_*.json") + glob.glob(f"{WORK}/figures/train_loss_*.png"):
        api.upload_file(path_or_fileobj=f, path_in_repo=f"logs/{os.path.basename(f)}", repo_id=MODEL_REPO)
    print("uploaded", MODEL_REPO, "(adapter + training logs)")
    if RUN_ABLATION:
        api.create_repo(ABL_REPO, repo_type="model", exist_ok=True)
        open(os.path.join(ABL_INFO["artifact"], "README.md"), "w").write(
            f"# Ablation adapter (no hard negatives / noise)\nSee {MODEL_REPO}. Trained on {ABL_INFO['n_train']} clean examples. For comparison only.\n")
        api.upload_folder(folder_path=ABL_INFO["artifact"], repo_id=ABL_REPO, repo_type="model")

    # ---- dataset: all splits, curated train, predictions, metrics, figures ----
    api.create_repo(DATA_REPO, repo_type="dataset", exist_ok=True)
    DATA_CARD = f"""---
license: cc-by-4.0
language: [en, hi]
task_categories: [text-generation, token-classification]
tags: [lexsi-hackathon, synthetic, upi, banking, hinglish, structured-output]
pretty_name: Indian bank & UPI SMS to JSON (synthetic, trilingual)
configs:
- config_name: default
  data_files:
  - {{split: train, path: data/train_curated.jsonl}}
  - {{split: dev, path: data/dev.jsonl}}
  - {{split: test_id, path: data/test_id.jsonl}}
  - {{split: test_ood, path: data/test_ood.jsonl}}
  - {{split: challenge, path: data/challenge_set.jsonl}}
---
# Indian bank & UPI SMS → JSON (English · Hindi · Hinglish)

Synthetic SMS with **exact** labels (a field is labelled only if its text survived generation), plus a 44-example **hand-written challenge set**.
`test_ood` uses message templates that never appear in training. Bank names are fictional; messages contain no links or phone numbers.

Schema: `{json.dumps(FIELDS)}`. Non-transactions (OTP, offers, reminders, future debits, balance info, money requests) → `is_transaction: false`, all other fields null.

Also included: `results/` (every model prediction, metrics, curation report from CuratorKIT, AuditKIT run records) and `figures/`.
"""
    open(f"{WORK}/README_dataset.md", "w").write(DATA_CARD)
    api.upload_file(path_or_fileobj=f"{WORK}/README_dataset.md", path_in_repo="README.md", repo_id=DATA_REPO, repo_type="dataset")
    for sub in ["data", "results", "figures"]:
        api.upload_folder(folder_path=f"{WORK}/{sub}", path_in_repo=sub, repo_id=DATA_REPO, repo_type="dataset")
    if os.path.isdir(f"{WORK}/curated"):
        api.upload_folder(folder_path=f"{WORK}/curated", path_in_repo="curatorkit_run", repo_id=DATA_REPO,
                          repo_type="dataset", allow_patterns=["**/*.json", "**/*.md", "**/*.txt", "**/*.jsonl"])
    print("uploaded", DATA_REPO)

### GGUF export for the free CPU demo
Merge the adapter into the base model, then convert with llama.cpp to `Q8_0` (no compilation needed). The demo Space runs this on 2 free vCPUs.

In [ ]:
GGUF_PATH = None
if MAKE_GGUF:
    try:
        free_gpu()
        from peft import PeftModel
        merged_dir = f"{WORK}/models/merged"
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cpu")
        art = FT_INFO["artifact"]
        m = PeftModel.from_pretrained(base, art).merge_and_unload() if os.path.exists(f"{art}/adapter_config.json") \
            else AutoModelForCausalLM.from_pretrained(art, torch_dtype=torch.float16, device_map="cpu")
        m.save_pretrained(merged_dir, safe_serialization=True); tok.save_pretrained(merged_dir)
        del m, base; gc.collect()
        if not os.path.isdir(f"{WORK}/llama.cpp"):
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", f"{WORK}/llama.cpp"], check=True)
        pip("-r", f"{WORK}/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt")
        GGUF_PATH = f"{WORK}/models/{PROJECT}-q8_0.gguf"
        r = subprocess.run([sys.executable, f"{WORK}/llama.cpp/convert_hf_to_gguf.py", merged_dir,
                            "--outfile", GGUF_PATH, "--outtype", "q8_0"], capture_output=True, text=True)
        print(r.stdout[-1500:], r.stderr[-1500:])
        if r.returncode != 0 or not os.path.exists(GGUF_PATH):
            GGUF_PATH = None
            print("⚠️ GGUF conversion failed; the Space will fall back to transformers on CPU (slower).")
        elif PUSH_TO_HUB:
            api.create_repo(GGUF_REPO, repo_type="model", exist_ok=True)
            api.upload_file(path_or_fileobj=GGUF_PATH, path_in_repo=os.path.basename(GGUF_PATH), repo_id=GGUF_REPO)
            open(f"{WORK}/README_gguf.md", "w").write(MODEL_CARD.replace("library_name: peft", "library_name: gguf"))
            api.upload_file(path_or_fileobj=f"{WORK}/README_gguf.md", path_in_repo="README.md", repo_id=GGUF_REPO)
            print("uploaded", GGUF_REPO)
    except Exception:
        traceback.print_exc()

In [ ]:
# ---- demo Space (Gradio, CPU) + collections ----
SPACE_FILES_NOTE = "The Space files (app.py, requirements.txt, README.md) are in the repo's space/ folder."
if PUSH_TO_HUB:
    try:
        space_dir = f"{WORK}/space"
        os.makedirs(space_dir, exist_ok=True)
        for f in ["smsgen.py", "scoring.py", "regex_baseline.py"]:
            shutil.copy(f"{WORK}/src/{f}", space_dir)
        open(f"{space_dir}/app.py", "w").write(SPACE_APP)
        open(f"{space_dir}/requirements.txt", "w").write(SPACE_REQS)
        open(f"{space_dir}/README.md", "w").write(SPACE_README.replace("__GGUF_REPO__", GGUF_REPO).replace("__MODEL_REPO__", MODEL_REPO).replace("__DATA_REPO__", DATA_REPO))
        api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", exist_ok=True)
        for k, v in {"GGUF_REPO": GGUF_REPO, "GGUF_FILE": os.path.basename(GGUF_PATH) if GGUF_PATH else "",
                     "LORA_REPO": MODEL_REPO, "BASE_MODEL": BASE_MODEL}.items():
            try: api.add_space_variable(SPACE_REPO, k, v)
            except Exception as e: print("space variable", k, e)
        if token and SPACE_ADD_TOKEN:   # only needed for the slow transformers fallback (gated base model)
            try: api.add_space_secret(SPACE_REPO, "HF_TOKEN", token)
            except Exception as e: print("space secret", e)
        api.upload_folder(folder_path=space_dir, repo_id=SPACE_REPO, repo_type="space")
        print("Space:", f"https://huggingface.co/spaces/{SPACE_REPO}")
    except Exception:
        traceback.print_exc()

    COLLECTIONS = {}
    try:
        cm = api.create_collection(title="Tiny Aya SMS extraction: models", namespace=HF_USERNAME,
                                   description="Lexsi hackathon T1-P5: LoRA + GGUF models", exists_ok=True)
        for rid in [MODEL_REPO] + ([GGUF_REPO] if GGUF_PATH else []) + ([ABL_REPO] if RUN_ABLATION else []):
            api.add_collection_item(cm.slug, item_id=rid, item_type="model", exists_ok=True)
        api.add_collection_item(cm.slug, item_id=SPACE_REPO, item_type="space", exists_ok=True)
        cd = api.create_collection(title="Tiny Aya SMS extraction: datasets", namespace=HF_USERNAME,
                                   description="Lexsi hackathon T1-P5: data, predictions, metrics, logs", exists_ok=True)
        api.add_collection_item(cd.slug, item_id=DATA_REPO, item_type="dataset", exists_ok=True)
        COLLECTIONS = {"models": f"https://huggingface.co/collections/{cm.slug}",
                       "datasets": f"https://huggingface.co/collections/{cd.slug}"}
        print(COLLECTIONS)
    except Exception:
        traceback.print_exc()

## 14. Render the Hugging Face article and the abstract from the real numbers
Writes `ARTICLE.md` and `ABSTRACT.md` into the working directory. Review, add your name, then paste into a Hugging Face article (huggingface.co/new-blog) and the submission form.

In [ ]:
def m(s, sp, k):
    return pct(RESULTS[(s, sp)]["metrics"].get(k)) if (s, sp) in RESULTS else "–"

ctx = {
    "TABLE_TEST_ID": TABLES["test_id"], "TABLE_TEST_OOD": TABLES["test_ood"], "TABLE_CHALLENGE": TABLES["challenge"],
    "LANG_TABLE_OOD": LANG_TABLES["test_ood"], "LANG_TABLE_CHALLENGE": LANG_TABLES["challenge"],
    "TAG_TABLE_OOD": TAG_TABLES["test_ood"], "TAG_TABLE_CHALLENGE": TAG_TABLES["challenge"],
    "SIG_TABLE": SIG_TABLE, "ROI_TABLE": ROI_TABLE,
    "N_TRAIN": str(len(train_rows)), "N_DEV": str(len(splits["dev"])), "N_TEST_ID": str(len(splits["test_id"])),
    "N_TEST_OOD": str(len(splits["test_ood"])), "N_CHALLENGE": str(len(splits["challenge"])),
    "FT_EM_ID": m("fine_tuned", "test_id", "exact_match"), "FT_EM_OOD": m("fine_tuned", "test_ood", "exact_match"),
    "FT_EM_CH": m("fine_tuned", "challenge", "exact_match"), "FS_EM_OOD": m("few_shot", "test_ood", "exact_match"),
    "FS_EM_CH": m("few_shot", "challenge", "exact_match"), "RX_EM_CH": m("regex", "challenge", "exact_match"),
    "ZS_EM_CH": m("zero_shot", "challenge", "exact_match"),
    "FT_HALL_CH": m("fine_tuned", "challenge", "hallucination_rate"), "FS_HALL_CH": m("few_shot", "challenge", "hallucination_rate"),
    "FT_NT_CH": m("fine_tuned", "challenge", "nontxn_f1"), "ABL_NT_CH": m("ft_no_robust_data", "challenge", "nontxn_f1"),
    "ABL_EM_CH": m("ft_no_robust_data", "challenge", "exact_match"), "ABL_HALL_CH": m("ft_no_robust_data", "challenge", "hallucination_rate"),
    "FT_SCHEMA_OOD": m("fine_tuned", "test_ood", "schema_valid"), "ZS_SCHEMA_OOD": m("zero_shot", "test_ood", "schema_valid"),
    "TRAIN_PATH": FT_INFO["path"], "TRAIN_MIN": str(FT_INFO.get("train_minutes")), "FT_FORMAT": FT_FORMAT,
    "CURATION": json.dumps(curation, ensure_ascii=False),
    "WRONG_FIELDS": ", ".join(f"{k} ({v})" for k, v in WRONG_FIELD_COUNTS.most_common(5)) or "none",
    "SELF_COST": f"{ROI['self_hosted']['usd_per_1M_sms']:,.2f}",
    "MONTHLY_SMS": f"{MONTHLY_SMS:,}", "SELF_MONTH": f"{ROI['self_hosted']['usd_per_month_on_demand']:,.2f}",
    "ALWAYS_ON_MONTH": f"{ROI['self_hosted']['usd_per_month_always_on_1gpu']:,.2f}",
    "API_MONTH": ", ".join(f"{k} ${v['usd_per_month']:,.2f}" for k, v in ROI["api"].items()),
    "GPU_HOURS": f"{ROI['self_hosted']['gpu_hours_per_month']:,.1f}", "TOK_S": f"{GEN_TOKENS_PER_SEC:.0f}",
    "TRAIN_COST": f"{ROI['self_hosted']['one_time_training_usd']:.2f}",
    "API_COST": ", ".join(f"{k} ${v['usd_per_1M_sms']:,.2f}" for k, v in ROI["api"].items()),
    "MODEL_REPO": MODEL_REPO, "GGUF_REPO": GGUF_REPO, "DATA_REPO": DATA_REPO, "SPACE_REPO": SPACE_REPO,
    "BASE_MODEL": BASE_MODEL, "LORA_R": str(LORA_R), "LORA_ALPHA": str(LORA_ALPHA), "LR": str(LR), "EPOCHS": str(EPOCHS),
}
def render_tpl(t):
    for k, v in ctx.items():
        t = t.replace("{{" + k + "}}", v)
    return t
open(f"{WORK}/ARTICLE.md", "w", encoding="utf-8").write(render_tpl(ARTICLE_TEMPLATE))
open(f"{WORK}/ABSTRACT.md", "w", encoding="utf-8").write(render_tpl(ABSTRACT_TEMPLATE))
abstract_words = len(render_tpl(ABSTRACT_TEMPLATE).split())
print("ARTICLE.md and ABSTRACT.md written | abstract words:", abstract_words, "(limit 500)")
print(render_tpl(ABSTRACT_TEMPLATE))